<a href="https://colab.research.google.com/github/davidanukege-boop/CapstoneProject/blob/main/Prostrate_Cancer_Image_Classification_Accuracy_Project_Interim_Draft_Anukege_David.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prostrate_Cancer_Image_Classification_Accuracy-Project_Anukege_David

## Problem Definition

Prostate cancer diagnostics currently face challenges in accurately classifying benign versus malignant cases, leading to potential over-diagnosis, under-diagnosis, and suboptimal treatment decisions. This project aims to address this by systematically comparing unimodal (image-only) and multimodal (image + clinical data) deep learning models to improve the precision of prostate cancer detection and diagnosis.

## Objective

The primary objective of the project is to compare the performance of unimodal and multimodal models in the detection and diagnosis of patient' prostate status.  if a patient's prostate is benign or maglinant.

This notebook focuses on Prostate Cancer Image Classification using various deep learning models. It involves extensive data preprocessing of medical images (MRI scans in .mha format) and accompanying clinical data. The project explores and compares different 3D deep learning architectures, including ResNet, EfficientNet, and Vision Transformers, both as image-only models and multimodal models that integrate clinical features. The models are evaluated using metrics like AUROC, Average Precision, Sensitivity, Specificity, and Dice Score, and statistical tests are performed to compare their efficacy. The overarching goal is to classify prostate cancer from medical images and understand the impact of different model types and multimodal data integration on performance.






## Data Dicitonary




### Data Dictionary

| Column Name        | Data Type | Description                                                              |
|--------------------|-----------|--------------------------------------------------------------------------|
| `patient_id`       | `Int64`   | Unique identifier for each patient.                                      |
| `study_id`         | `int64`   | Unique identifier for each MRI study/scan performed for a patient.     |
| `mri_date`         | `object`  | Date when the MRI scan was performed.                                    |
| `patient_age`      | `int64`   | Age of the patient at the time of the MRI scan.                          |
| `psa`              | `float64` | Prostate-Specific Antigen level (ng/mL), a key biomarker.              |
| `psad`             | `float64` | PSA Density (PSA / prostate volume), used for risk assessment.           |
| `prostate_volume`  | `float64` | Volume of the prostate gland (mL).                                       |
| `histopath_type`   | `object`  | Type of histopathological biopsy performed (e.g., MRBx for MRI-guided biopsy, SysBx for systematic biopsy). |
| `lesion_GS`        | `object`  | Gleason Score of the identified lesion(s), indicating cancer aggressiveness. |
| `lesion_ISUP`      | `object`  | ISUP Grade Group for the lesion(s), a simplified grading system for prostate cancer. |
| `case_ISUP`        | `int64`   | Overall ISUP Grade Group for the entire case, representing the most aggressive cancer found in the patient. |
| `case_csPCa`       | `object`  | Clinically significant Prostate Cancer status for the case (YES/NO).     |
| `center`           | `object`  | The medical center or institution where the patient data originated.     |


## Import Relevant Libraries

In [ ]:
!pip install pydicom SimpleITK picai_prep picai_baseline nnunet picai_eval imagehash dash

In [ ]:
# Initialize the random number generator
# Setting the seed value
SEED=1
from numpy.random import seed
seed(SEED)


from scipy.stats import skew, kurtosis

# to suppress warnings
import warnings
warnings.filterwarnings('ignore')

# for reading and manipulating data
import json
from scipy.ndimage import zoom
from pathlib import Path
from picai_eval import evaluate_folder
from tqdm import tqdm
import nibabel as nib # Added nibabel import
import imagehash
from PIL import Image
from scipy.stats import norm
import SimpleITK as sitk # Ensure SimpleITK is imported
from tensorflow.keras import layers, models
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.pipeline import FeatureUnion
from scipy.stats import skew, kurtosis
from scipy.stats import spearmanr
import cv2


import SimpleITK as sitk
import shutil
from functools import reduce
from ipywidgets import interact
from pathlib import Path
from picai_prep.examples.mha2nnunet.picai_archive import generate_mha2nnunet_settings
from picai_prep import MHA2nnUNetConverter
from picai_baseline.splits.picai import train_splits, valid_splits
import pydicom
import numpy as np
from numpy import mean
from numpy import std
import pandas as pd
#import pandas.util.testing as tm
import random
import os
import nnunet
import time
import os
import glob




# for visualization
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

# for testing model performance
from picai_prep import MHA2nnUNetConverter





from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score,\
roc_curve,confusion_matrix,roc_auc_score,precision_recall_curve,auc,average_precision_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error

import tensorflow as tf
# for building model
from sklearn.ensemble import BaggingClassifier,RandomForestClassifier,StackingClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn import tree
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from scipy.stats import spearmanr
from scipy.stats import wilcoxon
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score
from statsmodels.stats.contingency_tables import mcnemar
import scipy.stats

# for performing statistical analysis
import statsmodels.stats.api as sms
import statsmodels.tools.tools as add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from scipy.stats import tmean
from scipy.stats import randint as sp_randint
from scipy.stats import randint as sp_rand


# to split data
from sklearn.model_selection import train_test_split,StratifiedKFold,KFold,cross_val_score

# Normalizing the data
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from scipy.stats import zscore



# For Feature Selection and Regularization
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import LeaveOneOut
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import ClusterCentroids

# For Imputers
from sklearn.impute import KNNImputer
from sklearn.impute import SimpleImputer

# For Processing
from sklearn import preprocessing
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline

#For Deep Neural Network
import tensorflow as tf
tf.random.set_seed(SEED)
tf.experimental.numpy.random.seed(SEED)


from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras import optimizers
from tensorflow.keras.optimizers import Adam,SGD,RMSprop
#from keras.constraints import maxnorm
from sklearn.utils import class_weight
from sklearn.preprocessing import LabelEncoder
#from keras.wrappers.scikit_learn import KerasClassifier
from tensorflow.keras.utils import to_categorical

#
import cv2
# from google.colab.patches import cv2_imshow
from cv2 import cvtColor
from cv2 import cvtColorTwoPlane
from cv2 import imshow


from tensorflow.keras.layers import Dense, Dropout, Flatten #importing different layers
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Activation, Input, LeakyReLU,Activation
from tensorflow.keras import backend as K
from tensorflow.keras.utils import to_categorical #to perform one-hot encoding
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D,SpatialDropout2D
from tensorflow.keras.optimizers import RMSprop,Adam #optimiers for optimizing the model
from keras.callbacks import EarlyStopping,ReduceLROnPlateau #regularization method to prevent the overfitting
from keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import Sequential, Model,load_model
from tensorflow.keras import losses, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import precision_recall_fscore_support
from keras.applications.vgg16 import VGG16

In [ ]:
from google.colab import drive

drive.mount('/content/drive',force_remount=True)

# **Dataset Creation for Image Classification**

### Import Data and Analyzing Dataset

## Data Pre-Processing

#### Creating the dataframe

In [ ]:
# Reading the dataset from  Google Drive
images_path='/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/picai_data/'  # File containing Image files of the patient, the dependent variables
clinical_data_path = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/picai_labels/clinical_information/marksheet.csv"  # File containing the clinical information of the images, the independent variables
annotations_path="/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project//picai_labels/csPCa_lesion_delineations/human_expert/resampled" # File containing the labels of the images, the independent variables



In [ ]:
# Define a function to generate patient_ids and image paths based on the folder name
def data_label_generator(image_folder,path):
    image_paths = []
    patient_id= []
    for root, dirs, files in os.walk(image_folder):
        for file in files:
            if file.lower().endswith(( 't2w.mha','hbv.mha','adc.mha')):   #select only interesting images.
                image_path = os.path.join(root, file)
                patient_ids = os.path.basename(root) # Label is the subfolder name
                image_paths.append(image_path)
                patient_id.append(patient_ids)
            if file.lower().endswith(('nii.gz')):   #select only interesting images.
                  image_path = os.path.join(root, file)
                  patient_ids = "_".join(file.split("_")[:1])  # Label is the subfolder name
                  image_paths.append(image_path)
                  patient_id.append(patient_ids)

    # data_label_df= pd.DataFrame({'image_path': image_paths, 'label': labels})
    data_label_df= pd.DataFrame({f'{path}_path': image_paths,'patient_id': patient_id})
    return  data_label_df

In [ ]:
# 1. Load the Clinical Data (Marksheet)
# Ensure you have cloned the picai_labels repo

df_clinical= pd.read_csv(clinical_data_path)

#2. Generating image file path and patient_id
df_images=data_label_generator(images_path,'raw_image')


#3 Generating the annoated image and patient_id
df_annot=data_label_generator(annotations_path,'annot_image')


#3. Convert patient_id in df_images to numeric type
df_images['patient_id'] = pd.to_numeric(df_images['patient_id'], errors='coerce').astype('Int64')


#3. Convert patient_id in df_annot to numeric type
df_annot['patient_id'] = pd.to_numeric(df_annot['patient_id'], errors='coerce').astype('Int64')

#4. Convert patient_id in df_clinical to numeric type for consistency if needed, though it's already int64
df_clinical['patient_id'] = pd.to_numeric(df_clinical['patient_id'], errors='coerce').astype('Int64')


# 5. Combine DataFrames
# Merge clinical data with image mapping on patient ID
# List of DataFrames
dfs = [df_clinical, df_images, df_annot]

# Merge all DataFrames on a common column (e.g., 'key_col')
df= reduce(lambda left, right: pd.merge(left, right, on='patient_id', how='inner'), dfs)

# df = pd.merge(
#     df_clinical,
#     df_images,
#     left_on='patient_id',
#     right_on='patient_id',
#     how='inner')


# 4. View and Save Results
print(f"Combined {len(df)} cases.")
print(df.head())
df.to_csv("picai_combined_clinical.csv", index=False)

In [ ]:
df.head(5)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

#### Insight
1. There are 4335 patient cases, but about 1273 unique patient_ids.
2. There are duplicate items.
3. There are missing values for some features such as psa, psad, prostate_volume,histopath_type, lesion_ISUP, lesion_GS.

### Displaying Images

##### Printing Sample Images

In [ ]:
# Printing sample images
fig, axs = plt.subplots(1, 3, figsize=(25, 5))


for y,x in enumerate(df['patient_id'].unique()):
  b=len(df.query("patient_id== @x")['raw_image_path'].values)
  random_images = np.random.choice(b, 3, replace=False)

# Clear current subplots for each new patient
for ax_item in axs:
    ax_item.clear()

for u,z in enumerate(random_images):
    # u must not be greater than axis dimension
    if u >= len(axs):
        break

    image_query=df.query("patient_id == @x")['raw_image_path'].values[z]
    image_name=image_query.split('_')[-1]
    patient_id=image_query.split('/')[-2]
    image = sitk.ReadImage(image_query)
    img_array = sitk.GetArrayFromImage(image)

    # Adjust slicing based on image dimensionality
    slice_idx = 'N/A'
    if img_array.ndim == 3:
        # For 3D images, take the middle slice along the first axis (z-axis)
        slice_idx = img_array.shape[0] // 2
        slice_to_show = img_array[slice_idx, :, :]


    elif img_array.ndim == 2:
        # For 2D images, display the image directly
        slice_to_show = img_array
    else:
        print(f"Skipping image {image_query} with unexpected dimensions ({img_array.ndim}D).")
        continue # Skip this iteration if image has unsupported dimensions


    axs[u].imshow(slice_to_show, cmap='gray') #
    axs[u].set_title(f'Patient_ID:{patient_id}\nimage_type: {image_name}\nSlice {slice_idx}')
    axs[u].axis('off')
    # display_3D=interact(plot_slice, slice_idx=(0, img_array.shape[0]-1))

    # interact(plot_slice, slice_idx=(0, img_array.shape[0]-1),v=(0,2))
plt.tight_layout()
plt.show()

##### Display random samples of images with their labels, csPCa values

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(20, 25))

for y,x in enumerate(df[['patient_id', 'case_csPCa']].drop_duplicates()['case_csPCa'].sample(6)):
      image_query=df.query("case_csPCa == @x")['raw_image_path'].values[y]
      # Use SimpleITK to read .mha files
      image = sitk.ReadImage(image_query)
      img_array = sitk.GetArrayFromImage(image)

      # Take the middle slice if it's a 3D image
      if img_array.ndim == 3:
          slice_to_show = img_array[img_array.shape[0] // 2, :, :]
      elif img_array.ndim == 2:
          slice_to_show = img_array
      else:
          print(f"Skipping image {image_query} with unsupported dimensions ({img_array.ndim}D).")
          continue
      # img_slice.append(slice_to_show)
      if y < 2:

        axs[0, y].imshow(slice_to_show, cmap='gray')
        axs[0, y].axis('off')
        axs[0, y].set_title(f'Patient csPCa value:{x}')

      if y >= 2 and y < 4:
        y=y-2
        axs[1, y].imshow(slice_to_show, cmap='gray')
        axs[1, y].axis('off')
        axs[1, y].set_title(f'Patient csPCa value:{x}')
      if y >= 4:
        y=y-4
        axs[2, y].imshow(slice_to_show, cmap='gray')
        axs[2, y].axis('off')
        axs[2, y].set_title(f'Patient csPCa value:{x}')


plt.tight_layout()
plt.show()

In [ ]:
#
def plot_slice(slice_idx):
    plt.figure(figsize=(6,6))
    plt.imshow(img_array[slice_idx, :, :], cmap='gray')
    plt.title(f'Slice {slice_idx}')
    plt.axis('off')
    plt.show()

interact(plot_slice, slice_idx=(0, img_array.shape[0]-1))

In [ ]:
# Path to your PICAI .mha file
mha_file_path = '/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/picai_data/11416/11416_1001440_hbv.mha'

# Read the image
image = sitk.ReadImage(mha_file_path)

# Convert image to a numpy array (3D)
# PICAI images are usually (z, y, x) or (slices, height, width)
img_array = sitk.GetArrayFromImage(image)

print(f"Image shape: {img_array.shape}")
print(f"Spacing: {image.GetSpacing()}")

#### Insight
1. The visualization helps in understanding the visual characteristics of images associated with different case_csPCa labels.

2. The image core

In [ ]:
gmx=[]  # To hold the segmented and morphophed image
gix=[]
lower_green = (36,25,25)  # lower threshold  to determine the light green item in the image
upper_green = (86,255,255) # higher value to determine the dark green items in the image

# Define a target shape for all processed images to ensure consistency
# Adjust this target_shape based on your dataset's typical image dimensions
target_h = 256
target_w = 256
target_shape = (target_w, target_h) # OpenCV expects (width, height)

# Iterate over the DataFrame rows to ensure existing paths are processed
for i in np.arange(0, 3, 1):
    image_path = df['raw_image_path'][i]

    try:
        # Load the image using SimpleITK
        image_sitk = sitk.ReadImage(image_path)
        img_array_3d = sitk.GetArrayFromImage(image_sitk)

        # Extract a 2D slice if it's a 3D image (SimpleITK often gives Z, Y, X)
        if img_array_3d.ndim == 3:
            img_2d_raw = img_array_3d[img_array_3d.shape[0] // 2, :, :]
        elif img_array_3d.ndim == 2:
            img_2d_raw = img_array_3d
        else:
            print(f"Skipping {image_path}: Unsupported image dimension {img_array_3d.ndim}D.")
            continue

        # Normalize the 2D slice to 0-255 and convert to uint8 for OpenCV
        min_val = np.min(img_2d_raw)
        max_val = np.max(img_2d_raw)
        if max_val - min_val > 0:
            img_2d_normalized = ((img_2d_raw - min_val) / (max_val - min_val) * 255).astype(np.uint8)
        else:
            # Handle case of constant image (e.g., all black or all white)
            img_2d_normalized = np.zeros_like(img_2d_raw, dtype=np.uint8)

        # Convert grayscale image to BGR (required for HSV conversion)
        img_bgr = cv2.cvtColor(img_2d_normalized, cv2.COLOR_GRAY2BGR)

        # Convert BGR to HSV
        img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

        # Apply inRange for masking
        mask = cv2.inRange(img_hsv, lower_green, upper_green)

        struct = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(11,11))
        mask = cv2.morphologyEx(mask,cv2.MORPH_CLOSE,struct)

        # Apply the mask to the normalized 2D image
        result = cv2.bitwise_and(img_2d_normalized, img_2d_normalized, mask=mask)

        # Resize mask and result to the common target_shape
        mask_resized = cv2.resize(mask, target_shape, interpolation=cv2.INTER_AREA)
        result_resized = cv2.resize(result, target_shape, interpolation=cv2.INTER_AREA)

        gix.append(result_resized)
        gmx.append(mask_resized)

    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        continue

gmx=np.array(gmx)
gix=np.array(gix)


In [ ]:
#Displaying image shape after morphological transformation
gix.shape,gix.shape

#### Insight


### Enhanced Visualization of Segmented and Morphed Images

To ensure visibility and clarity, let's display all the processed images in `gmx` (segmented masks) and `gix` (morphed images), explicitly setting the display range (`vmin`, `vmax`) to maximize contrast for any non-zero pixels.

In [ ]:
# Get min/max values for consistent scaling across all gmx images
gmx_min = np.min(gmx)
gmx_max = np.max(gmx)

fig_gmx, axs_gmx = plt.subplots(1, gmx.shape[0], figsize=(6 * gmx.shape[0], 6))
fig_gmx.suptitle('All Segmented Images (gmx) - Enhanced Contrast', fontsize=16)

for i in range(gmx.shape[0]):
    axs_gmx[i].imshow(gmx[i], cmap='viridis', vmin=gmx_min, vmax=gmx_max)
    axs_gmx[i].set_title(f'Segmented Image {i}')
    axs_gmx[i].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# Get min/max values for consistent scaling across all gix images
gix_min = np.min(gix)
gix_max = np.max(gix)

fig_gix, axs_gix = plt.subplots(1, gix.shape[0], figsize=(6 * gix.shape[0], 6))
fig_gix.suptitle('All Morphed Images (gix) - Enhanced Contrast', fontsize=16)

for i in range(gix.shape[0]):
    axs_gix[i].imshow(gix[i], cmap='viridis', vmin=gix_min, vmax=gix_max)
    axs_gix[i].set_title(f'Morphed Image {i}')
    axs_gix[i].axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

#### Insight

## Exploratory Data Analysis

In [ ]:
# Define function for Countplot with Percentage on the graph
def count_graph(ax,z):
    plt.xticks(fontsize=12,rotation=60)
    for p in ax.patches:
        width = p.get_width()
        height= p.get_height()
        x, y = p.get_xy()
        ax.annotate('{:.2f}'.format(height), (p.get_x()+ width*0.7, (p.get_y() + height)), size=10,ha='center')

In [ ]:
# Define function for Countplot with Percentage on the graph
def count_graph(ax,z):
    plt.xticks(fontsize=12,rotation=60)
    for p in ax.patches:
        width = p.get_width()
        height= p.get_height()
        x, y = p.get_xy()
        ax.annotate('{:.2f}'.format(height), (p.get_x()+ width*0.7, (p.get_y() + height)), size=10,ha='center')

In [ ]:
# Defining a double graph--histplot, and boxplot
def boxcnt_graph(x_data,ccol,bcol,figsize=(20,10), bins=None):
    """ Boxplot and count_plot combined
    feature: 1-d feature array
    figsize: size of fig (default (9,8))
    bins: number of bins (default None / auto)
    """
    plt.figure(figsize=(20,10))
    ad=sns.countplot(x=x_data,color=ccol,data=df1);
    plt.xticks(fontsize=12,rotation=90)
    z=x_data
    for p in ad.patches:
        width = p.get_width()
        height= p.get_height()
        x, y = p.get_xy()
        ad.annotate('{:.2f}%'.format(100*height/len(z)), (p.get_x()+ width*0.7, (p.get_y() + height)), size=10,ha='center')
    plt.figure(figsize=(20,5))
    sns.boxplot(x=x_data,showmeans=True, color=bcol) # boxplot will be created and a star will indicate the mean value of the column


In [ ]:
# Defining a combo graph--countplot,histplot, and boxplot
def combo_graph(x_data,hcol,bcol,figsize=(20,10), bins=None):

    z=x_data

    """ Boxplot and histogram combined
    feature: 1-d feature array
    figsize: size of fig (default (9,8))
    bins: number of bins (default None / auto)
    """
    g, (ax_box,ax_dist) = plt.subplots(nrows = 2, # Number of rows of the subplot grid= 2
                                           sharex = True, # x-axis will be shared among all subplots
                                           gridspec_kw = {"height_ratios": (0.30,0.70)},
                                           figsize = figsize
                                           ) # creating the 2 subplots
    ad=sns.histplot(x=x_data, kde=True, ax=ax_dist, bins=bins,color=hcol) if bins else sns.histplot(x=x_data, kde=True, color=hcol,ax=ax_dist)
    for p in ad.patches:
        width = p.get_width()
        height= p.get_height()
        x, y = p.get_xy()
        ad.annotate('{:.2f}%'.format(100*height/len(z)), (p.get_x()+ width*0.7, (p.get_y() + height)), size=10,ha='center')
    sns.boxplot(x=x_data, ax=ax_box, showmeans=True, color=bcol) # boxplot will be created and a star will indicate the mean value of the column
     # For histogram

    ax_dist.axvline(np.mean(x_data), color='green', linestyle='--') # Add mean to the histogram
    ax_dist.axvline(np.median(x_data), color='black', linestyle='-') # Add median to the histogram
    plt.xticks(fontsize=12,rotation=90)

### Univariate Analysis

In [ ]:
# --- 1. SIMULATE OR EXTRACT DATA FROM NIFTI FILES ---
def extract_voxel_spacings(image_dir):
    """
    Scans a directory of NIfTI files and extracts voxel dimensions.
    Replace the mock loop below with your actual local file path loop.
    """
    records = []

    # Example pattern for PI-CAI images (e.g., matching all .nii.gz files)
    # Correctly search for multiple .mha files with specific suffixes
    file_paths = []
    for suffix in ["_t2w.mha", "_adc.mha", "_hbv.mha"]:
        file_paths.extend(glob.glob(os.path.join(image_dir, f"*{suffix}")))

    # # For demonstration, if no files are found, we populate a mock DataFrame
    # if len(file_paths) == 0:
    #     print("No files found at specified directory. Generating a mock cohort tracker for demonstration...")
    #     import numpy as np
    #     np.random.seed(42)
    #     # Simulating a mix of Siemens, Philips, and GE spacing profiles typical of PI-CAI
    #     mock_dx_dy = np.random.choice([0.3, 0.4, 0.5], size=200, p=[0.5, 0.4, 0.1])
    #     mock_dz = np.random.choice([3.0, 3.6, 4.0], size=200, p=[0.6, 0.3, 0.1])
    #     mock_seq = np.random.choice(['T2W', 'ADC', 'HBV'], size=200)

    #     return pd.DataFrame({
    #         'dx': mock_dx_dy,
    #         'dy': mock_dx_dy,
    #         'dz': mock_dz,
    #         'Sequence': mock_seq
    #     })

    for path in file_paths:
        filename = os.path.basename(path)
        try:
            # Use SimpleITK to load .mha files
            img_sitk = sitk.ReadImage(path)
            spacings = img_sitk.GetSpacing()

            # Determine sequence type based on PI-CAI naming standard (_0000=T2W, _0001=ADC, _0002=HBV)
            if "_t2w" in filename: seq = "T2W"
            elif "_adc" in filename: seq = "ADC"
            elif "_hbv" in filename: seq = "HBV"
            else: seq = "Unknown"

            records.append({
                'dx': round(spacings[0], 2),
                'dy': round(spacings[1], 2),
                'dz': round(spacings[2], 2),
                'Sequence': seq
            })
        except Exception as e:
            print(f"Skipping {filename} due to error: {e}")

    return pd.DataFrame(records)

# --- 2. RUN PIPELINE ---
# Point this to your actual raw PI-CAI images directory
for u,v in enumerate(df['raw_image_path'].sample(3)):
    IMAGE_DIRECTORY = os.path.dirname(v)
    df_spacing = extract_voxel_spacings(IMAGE_DIRECTORY)

    # Combine dx and dy into an in-plane string for easier value counts
    df_spacing['In-Plane (dx x dy)'] = df_spacing['dx'].astype(str) + " x " + df_spacing['dy'].astype(str)

    # --- 3. EXECUTE VALUE COUNTS ---
    print("\n=== FREQUENCY DISTRIBUTION: IN-PLANE RESOLUTION (dx x dy) ===")
    print(df_spacing['In-Plane (dx x dy)'].value_counts())

    print("\n=== FREQUENCY DISTRIBUTION: SLICE THICKNESS (dz) ===")
    print(df_spacing['dz'].value_counts())

    # --- 4. PLOT COUNT PLOTS ---
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    sns.set_theme(style="whitegrid")

    # Left Plot: In-Plane Resolution Count Plot stratified by Sequence
    sns.countplot(
        data=df_spacing,
        x='In-Plane (dx x dy)',
        hue='Sequence',
        ax=axes[0],
        palette='Set2'
    )
    axes[0].set_title("Distribution of In-Plane Resolution (dx, dy)", fontsize=12, fontweight='bold')
    axes[0].set_xlabel("Voxel Size (mm x mm)")
    axes[0].set_ylabel("Scan Count")

    # Right Plot: Slice Thickness (dz) Count Plot stratified by Sequence
    sns.countplot(
        data=df_spacing,
        x='dz',
        hue='Sequence',
        ax=axes[1],
        palette='Set2'
    )
    axes[1].set_title("Distribution of Axial Slice Thickness (dz)", fontsize=12, fontweight='bold')
    axes[1].set_xlabel("Voxel Thickness (mm)")
    axes[1].set_ylabel("Scan Count")

    plt.suptitle("PI-CAI Spatial Metadata Audit: Voxel Spacing Homogeneity", fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
def generate_extreme_value_table(image_dir):
    """
    Parses image headers and data arrays to track intensity distributions for .mha files.
    """
    records = []
    file_paths = []
    for suffix in ["_t2w.mha", "_adc.mha", "_hbv.mha"]:
        file_paths.extend(glob.glob(os.path.join(image_dir, f"*{suffix}")))

    # Fallback Simulation if no actual files are found
    if len(file_paths) == 0:
        print(f"No .mha files found in {image_dir}. Generating a mock cohort for demonstration...")
        # Simulating extreme values typical of PI-CAI data
        np.random.seed(42)
        mock_records = []
        for _ in range(3): # Simulate a few sequences
            seq = np.random.choice(['T2W', 'ADC', 'HBV'])
            data = np.random.normal(loc=500, scale=200, size=1000)
            if seq == 'ADC': # ADC values are typically lower
                data = np.random.uniform(200, 2200, size=1000)
            elif seq == 'HBV': # HBV can have higher intensity artifacts
                data = np.random.exponential(scale=150, size=1000)
                data[np.random.choice(1000, 10)] = np.random.uniform(5000, 18000)
            data = np.clip(data, 0, None) # Ensure non-negative intensities

            mock_records.append({
                'Sequence': seq,
                'Abs_Min': float(np.min(data)),
                'P1': float(np.percentile(data, 1)),
                'Mean': float(np.mean(data)),
                'P99': float(np.percentile(data, 99)),
                'Abs_Max': float(np.max(data))
            })
        return pd.DataFrame(mock_records)

    for path in file_paths:
        filename = os.path.basename(path)
        try:
            img_sitk = sitk.ReadImage(path) # Use SimpleITK to load .mha files
            vol = sitk.GetArrayFromImage(img_sitk) # Get numpy array from SimpleITK image

            # Identify sequence based on PI-CAI naming standard
            if "_t2w" in filename: seq = "T2W"
            elif "_adc" in filename: seq = "ADC"
            elif "_hbv" in filename: seq = "HBV"
            else: seq = "Unknown"

            records.append({
                'Sequence': seq,
                'Abs_Min': float(np.min(vol)),
                'P1': float(np.percentile(vol, 1)),
                'Mean': float(np.mean(vol)),
                'P99': float(np.percentile(vol, 99)),
                'Abs_Max': float(np.max(vol))
            })
        except Exception as e:
            print(f"Error parsing file {filename}: {e}")

    return pd.DataFrame(records)

# --- Execute Analysis ---
for u,v in enumerate(df['raw_image_path'].sample(3)):
    IMAGE_DIR = os.path.dirname(v)

    df_intensities = generate_extreme_value_table(IMAGE_DIR)

    # Aggregate across the cohort to find the global extreme bounds per sequence
    # Ensure df_intensities is not empty before grouping
    if not df_intensities.empty and 'Sequence' in df_intensities.columns:
        extreme_value_table = df_intensities.groupby('Sequence').agg({
            'Abs_Min': 'min',     # The lowest value observed across any patient
            'P1': 'median',       # Median 1st percentile cross-cohort
            'Mean': 'mean',       # Global cohort mean intensity
            'P99': 'median',      # Median 99th percentile cross-cohort
            'Abs_Max': 'max'      # The single absolute maximum value in the cohort
        }).round(2)

        print("\n=== COHORT EXTRACTION: GLOBAL EXTREME VALUE TABLE ===")
        print(extreme_value_table)
    else:
        print(f"\nNo intensity data found for {IMAGE_DIR} or 'Sequence' column missing. Skipping table generation.")

In [ ]:
# # --- 1. SIMULATE PI-CAI DISTRIBUTION COHORT ---
# # (Replace this block with: clinical_df = pd.read_csv("your_picai_meta.csv"))
# np.random.seed(42)
# n_samples = 500

# # Simulating classic prostate cohort characteristics
# sim_psa = np.random.lognormal(mean=1.9, sigma=0.6, size=n_samples) + 2.0
# # Add a few extreme clinical outliers typical of advanced csPCa
# sim_psa[np.random.choice(n_samples, 10)] = np.random.uniform(50, 120, 10)

# sim_volume = np.random.lognormal(mean=3.6, sigma=0.4, size=n_samples) + 10.0

# clinical_df = pd.DataFrame({
#     'PSA (ng/mL)': sim_psa,
#     'Prostate Volume (mL)': sim_volume
# })

clinical_df=df[['patient_id','patient_age','psad','prostate_volume','psa']].drop_duplicates()

# --- 2. STATISTICAL METRICS FUNCTION ---
def calculate_univariate_stats(df, column):
    data = df[column].dropna()
    stats = {
        'Mean': np.mean(data),
        'Median': np.median(data),
        'Min': np.min(data),
        'Max': np.max(data),
        'Skewness': skew(data),
        'Kurtosis': kurtosis(data)
    }
    return pd.Series(stats)

# --- 3. EXECUTE PLOTTING PIPELINE ---
fig, axes = plt.subplots(4, 2, figsize=(14, 20)) # Changed to 4 rows, 2 columns
sns.set_theme(style="whitegrid")

# features = ['PSA (ng/mL)', 'Prostate Volume (mL)']
features = ['patient_age','psad','prostate_volume','psa']

for i, col in enumerate(features):
    # Print descriptive summary to console
    print(f"\n=== Univariate Statistics for {col} ===")
    print(calculate_univariate_stats(df, col))

    # Left Column: Raw Histogram with KDE
    sns.histplot(data=clinical_df, x=col, kde=True, ax=axes[i, 0], color='steelblue', bins=40)
    axes[i, 0].axvline(clinical_df[col].median(), color='crimson', linestyle='--', linewidth=2, label=f'Median: {clinical_df[col].median():.2f}')
    axes[i, 0].axvline(clinical_df[col].mean(), color='darkgreen', linestyle='-', linewidth=2, label=f'Mean: {clinical_df[col].mean():.2f}')
    axes[i, 0].set_title(f'Raw Distribution of {col}', fontsize=12, fontweight='bold')
    axes[i, 0].set_ylabel('Patient Count')
    axes[i, 0].legend()

    # Right Column: Log-Transformed Histogram with KDE
    log_col = f'Log_{col.split()[0]}'
    clinical_df[log_col] = np.log1p(clinical_df[col])

    sns.histplot(data=clinical_df, x=log_col, kde=True, ax=axes[i, 1], color='teal', bins=40)
    axes[i, 1].axvline(clinical_df[log_col].median(), color='crimson', linestyle='--', linewidth=2)
    axes[i, 1].axvline(clinical_df[log_col].mean(), color='darkgreen', linestyle='-')
    axes[i, 1].set_title(f'Log-Transformed Distribution: ln({col.split()[0]} + 1)', fontsize=12, fontweight='bold')
    axes[i, 1].set_ylabel('Patient Count')

plt.suptitle("PI-CAI Exploratory Data Analysis: Univariate Clinical Features", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
combo_graph(df['prostate_volume'],'indigo','magenta')

In [ ]:
combo_graph(df['psa'],'indigo','magenta')

In [ ]:
combo_graph(df['patient_age'],'indigo','magenta')

In [ ]:
#Display the count of label classes
print (df[['patient_id', 'case_ISUP']].drop_duplicates()['case_ISUP'].value_counts())
print ()
# Display graphical distribution of label classess
plt.figure(figsize=(20,9))
ax=sns.countplot(x='case_ISUP',data=df[['patient_id', 'case_ISUP']].drop_duplicates())
count_graph(ax,df[['patient_id', 'case_ISUP']].drop_duplicates()['case_ISUP'])
plt.show()

In [ ]:
#Display the count of label classes
print (df[['patient_id', 'lesion_ISUP']].drop_duplicates()['lesion_ISUP'].value_counts())
print ()
# Display graphical distribution of label classess
plt.figure(figsize=(20,9))
ax=sns.countplot(x='lesion_ISUP',data=df[['patient_id', 'lesion_ISUP']].drop_duplicates())
count_graph(ax,df[['patient_id', 'lesion_ISUP']].drop_duplicates()['lesion_ISUP'])

#### Insights

1. The dataset is relatively balanced, though with the cat class having a single image more than the dog class.

In [ ]:
#Display the count of label classes
print (df[['patient_id', 'patient_age']].drop_duplicates()['patient_age'].value_counts())
print ()
# Display graphical distribution of label classess
plt.figure(figsize=(20,9))
ax=sns.countplot(x='patient_age',data=df[['patient_id', 'patient_age']].drop_duplicates())
count_graph(ax,df[['patient_id', 'patient_age']].drop_duplicates()['patient_age'])
plt.show()

In [ ]:
#Display the count of label classes
print (df[['patient_id', 'case_csPCa']].drop_duplicates()['case_csPCa'].value_counts())
print ()
# Display graphical distribution of label classess
plt.figure(figsize=(20,9))
ax=sns.countplot(x='case_csPCa',data=df[['patient_id', 'case_csPCa']].drop_duplicates())
count_graph(ax,df[['patient_id', 'case_csPCa']].drop_duplicates()['case_csPCa'])
plt.show()

### Pixel intensity distribution

In [ ]:
# Pixel Intensity for Original Images
num_images_to_sample = 200
pixel_intensities = []

print(f"Processing {num_images_to_sample} images to collect pixel data...")
for i, image_path in enumerate(df['raw_image_path'][:num_images_to_sample]):
    try:
        # Load the .mha or .nii.gz image using SimpleITK
        image = sitk.ReadImage(image_path)
        img_array = sitk.GetArrayFromImage(image)

        # Flatten the 3D volume into a 1D array of pixel values
        flattened_pixels = img_array.flatten()

        # Normalize pixel values to 0-1 range
        if flattened_pixels.size > 0:
            min_val = np.min(flattened_pixels)
            max_val = np.max(flattened_pixels)
            if (max_val - min_val) > 0:
                normalized_pixels = (flattened_pixels - min_val) / (max_val - min_val)
            else:
                normalized_pixels = np.zeros_like(flattened_pixels, dtype=np.float32) # All pixels are same value
            pixel_intensities.extend(normalized_pixels)
        else:
            print(f"Warning: Image {image_path} has no pixels.")

    except Exception as e:
        print(f"Skipping {image_path} due to error: {e}")

    if (i + 1) % 10 == 0:
        print(f"Processed {i+1} images...")

# Convert the list of pixel intensities to a NumPy array
pixel_intensities = np.array(pixel_intensities)

# Plot the histogram using Matplotlib
plt.figure(figsize=(20, 15))
# Only plot if there are pixel intensities collected
if pixel_intensities.size > 0:
    plt.hist(pixel_intensities, bins=256, range=[0, 1], color='gray', edgecolor='none')
    plt.title(f'Pixel Intensity Histogram for Sample of {len(df["raw_image_path"][:num_images_to_sample])} Medical Images')
    plt.xlabel('Normalized Pixel Intensity Value (0-1)')
    plt.ylabel('Frequency (Number of Pixels)')
    plt.xlim([0, 1])
    plt.grid(axis='y', alpha=0.75)
    plt.show()
else:
    print("No pixel intensities collected to plot histogram.")

#### Insight
For 200 samples, the pixel-intensity histogram exhibited a multimodal distribution with several peaks, with the tallest peaks at the right pixel ends,indicating that there are more bright images in the sample set than dark ones.

### Data Integrity Checks

#### Checking for Image Dimension

In [ ]:
def image_dims(image_path):
    try:
        # Use SimpleITK to read .mha files
        sitk_image = sitk.ReadImage(image_path)
        img_array = sitk.GetArrayFromImage(sitk_image)

        # If the image is 3D, take the middle slice for dimension calculation
        if img_array.ndim == 3:
            # Assuming (depth, height, width) order from SimpleITK
            slice_2d = img_array[img_array.shape[0] // 2, :, :]
        elif img_array.ndim == 2:
            slice_2d = img_array
        else:
            print(f"Error: Unsupported image dimension {img_array.ndim}D for {image_path}")
            return None, None

        height, width = slice_2d.shape
        return height, width

    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None, None

#### Insights
1. There are very few exact duplicate images in the datasets, whose deletion from the dataset can be ignored.

### Check Image Dimension

In [ ]:
# Number of images to sample
num_samples = 200

# Sample unique image paths from the DataFrame
sampled_image_paths = random.sample(df['raw_image_path'].unique().tolist(), min(num_samples, len(df['raw_image_path'].unique())))

sampled_dims = []
print(f"Processing dimensions for {len(sampled_image_paths)} sampled images...")
for path in sampled_image_paths:
    try:
        height, width = image_dims(path)
        sampled_dims.append({'height': height, 'width': width})
    except Exception as e:
        print(f"Error processing {path}: {e}")

sampled_df = pd.DataFrame(sampled_dims)

if not sampled_df.empty:
    plt.figure(figsize=(10, 8))
    plt.scatter(sampled_df['width'], sampled_df['height'], alpha=0.6)
    plt.xlabel("Image Width (pixels)")
    plt.ylabel("Image Height (pixels)")
    plt.title(f"Image Size Distribution for {len(sampled_image_paths)} Sampled Images")
    plt.grid(True)
    plt.show()
else:
    print("No image dimensions collected to plot.")

#### Insights
1. The pattern of the size for the image distribution and all images in each class is the same.
2. The image width varies with image height with many images in the two classes having lower width and heights less than 1000 pixels, and only few images in the two classes have heights and widths above 1000 pixels.

### Image Brightness

In [ ]:
def image_brightness(image_path):
    try:
        # Use SimpleITK to read .mha files
        sitk_image = sitk.ReadImage(image_path)
        img_array = sitk.GetArrayFromImage(sitk_image)

        # If the image is 3D, take the middle slice for brightness calculation
        if img_array.ndim == 3:
            # Assuming (depth, height, width) order from SimpleITK
            slice_2d = img_array[img_array.shape[0] // 2, :, :]
        elif img_array.ndim == 2:
            slice_2d = img_array
        else:
            print(f"Error: Unsupported image dimension {img_array.ndim}D for {image_path}")
            return None, None

        # Normalize pixel values to 0-255 range and convert to uint8 for OpenCV
        min_val = np.min(slice_2d)
        max_val = np.max(slice_2d)

        if max_val > min_val:
            # Scale to 0-255 and convert to uint8
            normalized_slice = ((slice_2d - min_val) / (max_val - min_val) * 255).astype(np.uint8)
        else:
            # Handle cases with uniform pixel values (e.g., all black or all white)
            normalized_slice = np.full(slice_2d.shape, 128, dtype=np.uint8) # Assign medium brightness

        # Convert the single-channel 2D image to a 3-channel image for cv2.cvtColor(BGR2HSV)
        # OpenCV's cvtColor expects a 3-channel image for BGR2HSV conversion.
        # Stacking the single channel three times creates a pseudo-BGR image where R=G=B.
        img_bgr = cv2.cvtColor(normalized_slice, cv2.COLOR_GRAY2BGR)

        # Calculate Brightness using HSV value channel
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        brightness_value = np.mean(hsv[:, :, 2])

        # Using the below assumed classification logic for Image brightness
        # Scale brightness from 0-255 to 0-100 percentage
        brightness_percentage = (brightness_value / 255) * 100

        if brightness_percentage <= 33:
            brightness_category = "Dark"
        elif brightness_percentage >= 66:
            brightness_category = "Bright"
        else:
            brightness_category = "Medium"

        return brightness_value, brightness_category

    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None, None

In [ ]:
brightness_category=[]       #list to hold the brightness category
brightness_values=[]         #list to hold the brightness values

for x in (df['case_csPCa'].unique()):
    dark_image=[]            #list for dark image
    bright_image=[]          #list for bright image
    medium_image=[]          #list for medium image
    temp_value=[]            #list to hold computed brightness values temporarily
    for i in (df.query("case_csPCa == @x")['raw_image_path'].sample(10).values):
        val,catg=image_brightness(i)
        if val is not None and catg is not None: # Ensure brightness calculation was successful
            temp_value.append(val)
            if catg=='Dark':
                dark_image.append(catg)
            elif catg=='Bright':
                bright_image.append(catg)
            elif catg=='Medium':
                medium_image.append(catg)

    # Only append if temp_value is not empty to avoid errors when np.round is called on an empty list
    if temp_value:
        temp_value_dict={'case_csPCa': x,
                                'Brightness_values': np.round(temp_value,2)}
        brightness_values.append(temp_value_dict)

    brightness_dict={'case_csPCa': x,
                'Dark': len(dark_image),
                'Bright':len(bright_image),
                'Medium': len(medium_image)}
    brightness_category.append(brightness_dict)


brightness_df=pd.DataFrame(brightness_category)
brightness_df            #dataframe for showing the count of images that are either dark, bright, or medium

In [ ]:
ax=brightness_df.plot.bar(x='case_csPCa', y=['Dark','Bright','Medium'],figsize=(20, 15))
for container in ax.containers:
    ax.bar_label(container,fontsize=10)
plt.title('Case csPCa vs Brightness Category')
plt.xlabel('Case csPCa')
plt.ylabel('Brightness Category')
plt.show()

In [ ]:
# Plotting Brightness Values for Each Label using Box Plot

b_label=[]   # empty list for brightness labels
b_means=[]    # empty list for brightness aggregate means

# convert brightness_values to a dataframe
b_values_df=pd.DataFrame(brightness_values)

# Explode the individual columns in b_values
b_values_df=b_values_df.explode('case_csPCa')
b_values_df=b_values_df.explode('Brightness_values')

for i in b_values_df['case_csPCa'].unique():
    b_label.append(i)
    value=b_values_df[b_values_df['case_csPCa']==i]['Brightness_values'].values
    mean=np.round(value.mean(),2)
    b_means.append(mean)
mean_df= pd.DataFrame({'case_csPCa':b_label, 'Brightness Mean': b_means})
print (mean_df)
print ('\n\n')
plt.figure(figsize=(25, 15))
ax=sns.boxplot(x='case_csPCa', y='Brightness_values', showmeans=True,data=b_values_df)
plt.title('Box Plot of Brightness Means', fontsize=20)
plt.ylabel('Value',fontsize=20)
plt.xlabel('case_csPCa',fontsize=20)
plt.show()

#### Insights
1. The images with medium brightness in each class have the highest number, followed by images with bright brightness, and then, images with dark brightness.
2. There are few or no upper outliers for image brightness values in most classes.
3. Each image class has lower outliers for image brightness values.

### Image Aspect Ratio

In [ ]:
def image_aspect_ratio(image_path):
    try:
        # Use SimpleITK to read .mha files
        sitk_image = sitk.ReadImage(image_path)
        img_array = sitk.GetArrayFromImage(sitk_image)

        # If the image is 3D, take the middle slice for aspect ratio calculation
        if img_array.ndim == 3:
            # Assuming (depth, height, width) order from SimpleITK
            slice_2d = img_array[img_array.shape[0] // 2, :, :]
        elif img_array.ndim == 2:
            slice_2d = img_array
        else:
            print(f"Error: Unsupported image dimension {img_array.ndim}D for {image_path}")
            return None, None

        # Get height and width from the 2D slice
        height, width = slice_2d.shape

        # Calculate Aspect Ratio
        aspect_ratio = width/height

        # Using the below assumed classification logic for image aspect ratio
        if abs(aspect_ratio - 1.0) < 0.1:
            aspect_category = "Square"
        elif aspect_ratio > 1.5:
            aspect_category = "Wide"
        else:
            aspect_category = "Tall/Other"

        return aspect_ratio, aspect_category

    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None, None

In [ ]:
aspect_category=[]       #empty list to hold the aspect ratio category
aspect_ratio_values=[]   #empty list to hold the aspect_ration category

for x in (df['case_csPCa'].unique()):
    square_image=[]
    wide_image=[]
    tall_image=[]
    temp_value=[]
    for i in (df.query("case_csPCa == @x")['raw_image_path'].sample(10).values):
        val,catg=image_aspect_ratio(i)
        if val is not None and catg is not None: # Add check for None values
            temp_value.append(val)

            if catg=='Square':
                square_image.append(catg)
            else:
                if catg=='Wide':
                  wide_image.append(catg)
                else:
                    if catg=='Tall/Other':
                         tall_image.append(catg)

    # Use the category 'x' directly instead of re.split on 'i'
    aspect_values={'case_csPCa': x,
                  'Aspect Ratio': temp_value
                 }


    aspect_ratio_values.append(aspect_values)

    aspect_catg={'case_csPCa': x, # Use x directly
                'Square': len(square_image),
                'Wide':len(wide_image),
                'Tall/Other': len(tall_image)
                    }
    aspect_category.append(aspect_catg)

aspect_ratio_df=pd.DataFrame(aspect_category)
aspect_ratio_df               #shows the count of images that are either square, wide, or tall/order

In [ ]:
# Plot a vertical bar chart
ax=aspect_ratio_df.plot.bar(x='case_csPCa', y=['Square','Wide','Tall/Other'],figsize=(20, 15))
for container in ax.containers:
    ax.bar_label(container,fontsize=10)
plt.title('Label vs Aspect Ratio Category')
plt.xlabel('case_csPCa')
plt.ylabel('Aspect Ratio Category')
plt.show()

In [ ]:
# Plotting aspect ratio  values for each label using box plot

a_label=[]   # empty list for aspect ratio labels
a_means=[]    # empty list for aspect ratio aggregate means

# convert aspect_ratio_values to a dataframe
a_values_df=pd.DataFrame(aspect_ratio_values)

# Explode the individual columns in b_values
a_values_df=a_values_df.explode('case_csPCa')
a_values_df=a_values_df.explode('Aspect Ratio')

for i in a_values_df['case_csPCa'].unique():
    a_label.append(i)
    value=a_values_df[a_values_df['case_csPCa']==i]['Aspect Ratio'].values
    mean=np.round(value.mean(),2)
    a_means.append(mean)
a_mean_df= pd.DataFrame({'case_csPCa':a_label, 'Aspect Ratio Mean': a_means})
print (a_mean_df)
print ('\n\n')
plt.figure(figsize=(25, 15))
ax=sns.boxplot(x='case_csPCa', y='Aspect Ratio', showmeans=True,data=a_values_df)
plt.title('Box Plot of Aspect Ratio Means', fontsize=20)
plt.ylabel('Value',fontsize=20)
plt.xlabel('Label',fontsize=20)
plt.show()

#### Insights
1. Images in the wide category have the highest number of images in each class. This is followed by images in the tall category, and then images in the square category.
   

###  Bivariate Analysis

In [ ]:
# --- 1. SIMULATE MULTIMODAL PAIRED METRICS ---
np.random.seed(101)
n_cases = 300

# Group 0: Benign / Low-Risk (Higher ADC, Lower PSAD)
mean_adc_benign = np.random.normal(loc=1300, scale=200, size=180)
psad_benign = np.random.lognormal(mean=-2.3, sigma=0.4, size=180)
labels_benign = np.zeros(180)

# Group 1: Clinically Significant Cancer (Lower ADC, Higher PSAD)
mean_adc_malignant = np.random.normal(loc=750, scale=180, size=120)
psad_malignant = np.random.lognormal(mean=-1.4, sigma=0.5, size=120)
labels_malignant = np.ones(120)

# Combine into clinical summary dataframe
analysis_df = pd.DataFrame({
    'Mean_Lesion_ADC': np.concatenate([mean_adc_benign, mean_adc_malignant]),
    'PSA_Density': np.concatenate([psad_benign, psad_malignant]),
    'csPCa_Status': np.concatenate([labels_benign, labels_malignant]).astype(int)
})

# --- 2. EXECUTE BIVARIATE PLOT ---
plt.figure(figsize=(20, 15))
sns.set_theme(style="ticks")

# Map status to readable string names for presentation
analysis_df['Diagnosis'] = analysis_df['csPCa_Status'].map({0: 'Benign / Low-Grade', 1: 'csPCa (ISUP >= 2)'})

sns.scatterplot(
    data=analysis_df,
    x='Mean_Lesion_ADC',
    y='PSA_Density',
    hue='Diagnosis',
    style='Diagnosis',
    palette={'Benign / Low-Grade': 'forestgreen', 'csPCa (ISUP >= 2)': 'crimson'},
    alpha=0.8,
    s=70
)

plt.yscale('log') # Log scale because PSAD is naturally right-skewed
plt.title("Bivariate Spatial-Clinical Mapping: Voxel Intensity vs. Biomarker Load", fontsize=13, fontweight='bold')
plt.xlabel("Mean ADC Intensity Value within ROI ($mm^2/s$)", fontsize=11)
plt.ylabel("PSA Density ($ng/mL^2$) [Log Scale]", fontsize=11)
plt.grid(True, which="both", ls="--", alpha=0.5)

# Add clear diagnostic thresholds frequently cited in radiology literature
plt.axvline(x=900, color='gray', linestyle=':', linewidth=1.5, label='Typical ADC Restriction Boundary')
plt.axhline(y=0.15, color='gray', linestyle='-.', linewidth=1.5, label='Standard Biopsy Cutoff (0.15)')

plt.legend(frameon=True, facecolor='white', edgecolor='lightgray')
plt.tight_layout()
plt.show()

In [ ]:
#Display the count of label classes
# Display graphical distribution of label classess
plt.figure(figsize=(20,15))
ax=sns.countplot(x='patient_age',data=df[['patient_id', 'patient_age']].drop_duplicates())
count_graph(ax,df[['patient_id', 'patient_age','case_csPCa']].drop_duplicates()['case_csPCa'])
plt.show()

In [ ]:
# Display graphical distribution of label classess
plt.figure(figsize=(20,15))
ax=sns.countplot(x='psad',hue='case_csPCa',color='firebrick',data=df[['patient_id', 'psad','case_csPCa']].drop_duplicates());
count_graph(ax,df[['patient_id', 'psad','case_csPCa']].drop_duplicates()['case_csPCa'])
plt.show()

In [ ]:
plt.figure(figsize=(20,15))
ax=sns.countplot(x='patient_age',hue='case_csPCa',color='firebrick',data=df[['patient_id', 'patient_age','case_csPCa']].drop_duplicates());
count_graph(ax,df[['patient_id', 'patient_age','case_csPCa']].drop_duplicates()['case_csPCa'])
plt.show()

In [ ]:
# Observing the Correlation among Numerical Variables
# Showing the correlation using heatmap
plt.figure(figsize=(20,15))
# Select only numerical columns for correlation calculation
numeric_df = df.select_dtypes(include=np.number).drop(columns=['patient_id','study_id'])
ax=sns.heatmap(numeric_df.corr(),annot=True);
plt.title('Correlation Among Numerical Variables');

In [ ]:
plt.figure(figsize=(20,15))
ax=sns.pairplot(numeric_df,diag_kind='kde');
plt.show()

#### Brightness vs Aspect Ratio for Each Class

In [ ]:
# Merging the dataframes from Brightness and Aspect Ratio Values, b_values_df, and a_values_df, on label
brightness_aspect_df=pd.merge(a_values_df,b_values_df,on='case_csPCa')
brightness_aspect_df.sample(5)              #displaying sample of 5 from the new dataframe

In [ ]:
#Display the relationship between Image Brightness and Aspect Ratio
plt.figure(figsize=(20,15))
sns.pairplot(brightness_aspect_df, hue='case_csPCa',height=3, aspect=1.5)
plt.show()

#### Insights
1. The brightness for images within each class is normally distributed, with all classes having a close mean and with an almost similar bell-shaped pattern.
2. The aspect-ratio for images within each class is normally distributed, with each class having a different mean, but with an almost similar bell-shaped pattern.
3. The relationship among images of different classes for the brightness values and the aspect ratio values varies, as multiple images with the same aspect ratio have varying brightness, and vice versa.
4. There is no apparent linear relationship between image brightness and aspect ratio.

# Model Building

### Model Evaluation Criteria

Problem: Predict correctly the class of food from an image.

The food recognition system, Nutriscan, needs to correctly predict the class of food contained in an image based on the following scenarios:

a) Determine the class of food majorly present in an image, and provide helpful information about the nutrition benefits of such a class of food to the users

b) Determine if an image does not contain any class of food that may be of any nutritional benefit to the users.

Therefore, the desired model will be such that:

i) If an image is predicted to contain a class of food with nutritional benefits, and it turns out that such a class of food is present in the image - True positive; this eases nutritional tracking, promotes healthier eating habits, and support users in achieving their dietary and wellness goals with minimal effort; a gain in the use of resources since the intended purpose of the food recognition system is acheived; improves customer satisfaction.

ii) If an image is predicted to contain a class of food with nutritional benefits, and it turns out that such a class of food is not present in the image - False positive; this is a significant loss of opportunity to achieve the users' dietary goals; waste of system resources and may increase customer dissatisfaction with the use of the system.

iii) If an image is predicted not to contain a class of food with nutritional benefits, and it turns out that such a class of food is present in the image -- False negative; though the users do get the nutritional benefits of the class of food anyway, but may not help in improving their dietary and wellness goals; may cause nutritional issues for the users if the wrong class of food is consumed reduces customer satisfaction, etc.

iv) If an image is predicted not to contain a class of food with nutritional benefits, and it turns out that such a class of food is not present in the image - True negative; no waste of system resources, no wrong information passed to the users, improves customer satisfaction with the system, etc.

Therefore, the desired outcome is to effectively determine whether a class of food with nutritional benefits is present in an image.

Thus, the goal is to reduce false positives and false negatives simultaneously.

a. To achieve item #(iii) above, a model that can increase true positives and reduce false negatives, that is, increase recall, will be appropriate

b. To achieve item #(ii) above, a model that can increase true positives and reduce false positives, that is, increase precision, will be appropriate

On the other hand, having a model that can give a balance between precision and recall, with high accuracy, is preferred. The lower the false negatives and false positives, the more accurately the outcome is predicted at all times.

Therefore, a high value for accuracy, recall, precision, and f1_score will be desirable.




### Model Evaluation Criterion

#### Problem-Predict if an existing customer will attrite the credit card services or not
1. The organization, AUSPG, needs to decide based on the following scenarios:

a). Stop the increase in exit rate from their bank in order to maintain or increase revenue.   

b)  Improve services perhaps by giving some incentives in terms of credit days, interest rate reduction, etc.

c. The desire outcome is to determine customers that will exit from the bank, and to determine variables attributing to them leaving

Therefore, the desired model will be such that:

i)  If the bank predict a customer 'will not exit' and  he or she 'does not exit'--True positive- revenue is sustained or increased.

ii) If they predict a customer 'will not exit' and he or she 'does exit' despite the incentives given--loss of opportunity to have made interest on credit, or giving the money to another customer that will not exit, reduction in revenue--False Positives

iii) If they predict a customer 'will exit' and he or she 'does not exit' the services--False Negative--Revenue gain

iv) If they  predict a customer 'will exit' and he or she 'does exit' the services. True Negative-- no loss in revenue nor loss in opportunity


2. The ultimate goal of the bank will be to minimize the number of customers exiting from the bank by being able to rightly predict the customer that will exit. This will enable them maintain or increase revenue. Thus, the goal is to reduce false positives, and false negatives.   


3. To achieve items#1c(iii) above, the bank will want to increase true positive and reduce false negatives,that is increase recall.

4. To achieve items#1c(ii) above, the bank will want to increase true positive and reduce false positives,that is increase precision

5. Thus, the bank will like to have a balance between precision and recall.The lower the false negatives and false positives, the lower the number of customers exiting the bank and the better the revenue.


6. A high value for recall, precision and f1_score will be desireable.

### Defining functions for model evaluations

In [ ]:
# Define a function for displaying the model performanc--tables and graph
def model_perf_graph(model,model_history,title):
    t_loss,t_accuracy=model.evaluate(X_train, Y_train)
    tt_loss,tt_accuracy=model.evaluate(X_test, Y_test)
    v_loss,v_accuracy=model.evaluate(x_val, y_val)
    print ('\n\n')
    print (f'\033[1m{str(title)} Performance Metrics\033[0m')
    print ()
    print (f'Train Accuracy:{t_accuracy}')
    print ()
    print (f'Test Accuracy:{tt_accuracy}')
    print ()
    print (f'Validation Accuracy:{v_accuracy}')
    print ('\n\n')
    # Plotting the training vs validation accuracy at different epochs
    print(model_history.history.keys())
    hist  = pd.DataFrame(model_history.history)
    hist['epoch'] = model_history.epoch
    plt.figure(figsize=(20,15))
    plt.title(f'{str(title)} Training vs Validation Accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.plot(hist['accuracy'])
    plt.plot(hist['val_accuracy'])
    plt.legend(("training" , "validation") , loc =0);
    plt.show()
    print ('\n\n')

In [ ]:
# Defining a function for model performance evaluation

def model_perf(model,x,y):
    perf_dict={}
    model_pred=model.predict(x)
    model_pred=np.argmax(model_pred, axis=1)
    y_true=np.argmax(y, axis=1)
    accuracy = np.round(metrics.accuracy_score(np.array(y_true), model_pred),2)
    precision = np.round(precision_score(y_true, model_pred, average='micro'),2)
    recall = np.round(recall_score(y_true, model_pred, average='micro'),2)
    f1= np.round(f1_score(y_true, model_pred, average='micro'),2)
    # Create a dictionary for performance outputs
    perf_dict ={ 'Accuracy' : accuracy,
                 'Precision' : precision,
                 'Recall' : recall,
                 'f1_score' : f1
               }
    # Convert dictionary to a dataframe
    df=pd.Series(perf_dict)
    return df

In [ ]:
# Define a function for displaying the model's confusion matrix

def model_conf_matrix(model,x,y,title):
    test_pred = model.predict(x)
    test_pred_classes = np.argmax(test_pred, axis=1)
    y_true=np.argmax(y, axis=1)
    matrix = confusion_matrix(np.array(y_true), test_pred_classes)
    # Confusion matrix normalized per category true value
    matrix1 = matrix/np.sum(matrix, axis=1)
    plt.figure(figsize=(25,15))
    ax=sns.heatmap(matrix1,cmap=plt.cm.Spectral_r, xticklabels=class_labels, yticklabels=class_labels, annot=True);
    ax.set_title(title, fontsize=30)
    ax.set_ylabel('Actual', fontsize=30)
    ax.set_xlabel('Predicted', fontsize=30);
    plt.show()
    print(f'=== Model Classification Report ===')
    print (classification_report(y_true,test_pred_classes))
    #aa=pd.DataFrame(a)
    # The weighted f1-Score for this model
    print ('\n')
    print (f'The weighted f1-Score of this model, model:{f1_score(np.array(y_true), test_pred_classes, average='weighted')}')
    print ('\n\n')

In [ ]:
# Defining a function for TSNE-visualization

# 1. Load the pre-trained transfer model
def tsne_visualization(model_inst,target_shape):
        model = model_inst

        # Ensure the model is built by passing dummy input if not already built
        # This is crucial if the kernel was restarted or cells were run out of order
        if not model.built:
            dummy_input = tf.zeros((1, 224, 224, 3))
            _ = model(dummy_input)


        input_data_for_prediction = X_train

        # Print model summary for reference to identify correct layer names
        print("\nModel Summary for reference:")
        model.summary()

        # Example of extracting a specific intermediate layer's activations
        # Correcting layer name to match the model summary output for trf_model (e.g., 'conv2d')
        fnames=['1st','global']
        lnames=['last','dense']
        last_names=[]
        for i, layer in enumerate(model.layers):
            if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense,tf.keras.layers.GlobalAveragePooling2D)):
                if any(name in layer.name for name in fnames):
                    first_layer_name = layer.name
                if any (name in layer.name for name in lnames):
                    last_names.append(layer.name)
                    last_layer_name = last_names[0]

        layer_name_specific = first_layer_name # Assuming the fist Conv2D layer named 'first_layer_name' from the summary

        try:
            # Create a functional model that extracts output from the specified intermediate layer
            intermediate_layer_model = tf.keras.Model(inputs=model.inputs[0], outputs=model.get_layer(layer_name_specific).output)

            # Get the activations (feature map) for the input data
            activations_specific_layer = intermediate_layer_model.predict(input_data_for_prediction, verbose=0)

            # Calculate the sparsity (percentage of zero activations) for this specific layer
            total_activations_specific = np.prod(activations_specific_layer.shape)
            zero_activations_specific = np.sum(activations_specific_layer == 0)
            sparsity_percentage_specific = (zero_activations_specific / total_activations_specific) * 100
            mean_val = np.mean(activations_specific_layer)
            std_dev_val = np.std(activations_specific_layer)

            print(f"\nActivations for layer '{layer_name_specific}':")
            print(f"  Shape: {activations_specific_layer.shape}")
            print(f"  Total activations: {total_activations_specific}")
            print(f"  Number of zero activations: {zero_activations_specific}")
            print(f"  Feature sparsity percentage: {sparsity_percentage_specific:.2f}%")
            print(f"  Mean Val: {mean_val}")
            print(f"  Standard Deviation: {std_dev_val}")


        except ValueError as e:
            print(f"\nError accessing layer '{layer_name_specific}': {e}")
            print("Please check the model summary above for correct layer names.")


        # Loop through all layers to get layer-specific sparsity for 'conv' and 'dense' layers
        print("\nSparsity for all convolutional and dense layers:")

        # Create a new Input tensor for the activation_model graph construction
        functional_input_tensor = tf.keras.Input(shape=(target_shape))
        functional_intermediate_outputs = []
        current_layer_output = functional_input_tensor

        for layer in model.layers:
            current_layer_output = layer(current_layer_output)
            functional_intermediate_outputs.append(current_layer_output)

        activation_model = tf.keras.Model(inputs=functional_input_tensor, outputs=functional_intermediate_outputs)
        all_activations = activation_model.predict(input_data_for_prediction, verbose=0)

        if not isinstance(all_activations, list):
            all_activations = [all_activations]

        # Iterate through each layer of the trf_model to get layer-specific sparsity
        for i, layer in enumerate(model.layers):
            if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.Dense,tf.keras.layers.GlobalAveragePooling2D)):
                layer_activations = all_activations[i]
                total = np.prod(layer_activations.shape)
                zeros = np.sum(layer_activations == 0)
                percent_zero = (zeros / total) * 100
                mean_val = np.round(np.mean(layer_activations),2)
                std_dev_val = np.round(np.std(layer_activations),2)
                print(f"- {layer.name} (shape: {layer_activations.shape}):, sparsity_percentage:{percent_zero:.2f}% zero activations, mean val:{mean_val}, standard devication:{std_dev_val}")

            elif isinstance(layer, tf.keras.Sequential):
                print(f"- Skipping composite layer {layer.name} for direct sparsity calculation.")
            else:
                print(f"- Skipping {layer.name} (not a Conv2D, Dense, or GlobalAveragePooling2D layer for direct sparsity analysis)")


        print ('\n\n')
        img_data = input_data_for_prediction # Use the same input data as for sparsity

        labels=[class_labels[idx] for idx in np.argmax(Y_train[:img_data.shape[0]], axis=1)]

        # --- FIX FOR T-SNE ERROR --- #
        # Define the layer from which to extract features for t-SNE
        tsne_feature_layer_name = last_layer_name # Using the last layer name for feature extraction for t-SNE
        tsne_feature_model = tf.keras.Model(inputs=model.inputs[0], outputs=model.get_layer(tsne_feature_layer_name).output)

        try:
            # Create a functional model that outputs the features from the chosen layer
            tsne_feature_model = tf.keras.Model(inputs=model.inputs[0], outputs=model.get_layer(tsne_feature_layer_name).output)

            # Get the features for the input data
            features_for_tsne = tsne_feature_model.predict(img_data, verbose=0)
            print (f'Shape of features_for_tsne:{features_for_tsne.shape}')

            # flatten the features to 2D array
            features_for_tsne = features_for_tsne.reshape(features_for_tsne.shape[0], -1)

            # TSNE Visualization

            print(f"Running t-SNE on features from layer '{tsne_feature_layer_name}' of shape: {features_for_tsne.shape}...")
            tsne = TSNE(n_components=2, perplexity=min(30,features_for_tsne.shape[0] - 1), random_state=42, init='pca', learning_rate='auto')
            tsne_results = tsne.fit_transform(features_for_tsne)

            # Plot the results
            plt.figure(figsize=(20, 15))
            sns.scatterplot(
                x=tsne_results[:, 0], y=tsne_results[:, 1],
                hue=labels,
                palette=sns.color_palette("hsv", n_colors=len(np.unique(labels))),
                legend="full",
                alpha=0.7
                  )
            plt.title('t-SNE visualization of feature space by class')
            plt.xlabel('t-SNE component 1')
            plt.ylabel('t-SNE component 2')
            plt.legend(title='Class')
            plt.show()

        except ValueError as e:
            print(f"\nError accessing layer '{tsne_feature_layer_name}' for t-SNE: {e}")
            print("Please check the model summary for correct layer names or choose a different layer.")
        except Exception as e:
            print(f"\nAn unexpected error occurred during t-SNE visualization: {e}")


In [ ]:
# Defining Function for KFold Cross-Validation
#KFold CV with Boxplot
def kfold_cv(model_inst,X,y):
    kfcv_results=[]
    kf_cv=KFold(n_splits=5,shuffle=True,random_state=SEED)     #Setting number of splits equal to 5


    for train, test in kf_cv.split(X, y):
            # 1. Define model architecture inside the loop
            model = model_inst

            # 2. Train
            model.fit(X[train], y[train], epochs=10, batch_size=32, verbose=0)

            # 3. Evaluate
            kfcv_result = model.evaluate(X[test], y[test], verbose=0)
            kfcv_results.append(kfcv_result[1]) # Accuracy

            # print(f"Average accuracy: {np.mean(kfcv_results)}")

    #Plotting boxplots for CV scores of model defined above
    plt.figure(figsize=(10,9))
    plt.boxplot(kfcv_results,showmeans=True)
    plt.show()

    print ()
    print ('\033[1m Kfold Accuracy Results')
    print("\033[0m Mean Score:{} \n Standard Deviation:{}".format(np.mean(kfcv_results) * 100,std(kfcv_results)))

### Image Processing using nnU-net Model

#### nnU-Net

In [ ]:
# Generate mha2nnunet_settings.json" for 3D nnU-net
output_path = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net"
os.makedirs(output_path, exist_ok=True)

# generate_mha2nnunet_settings(
#     archive_dir=images_path,
#     annotations_dir=annotations_path,
#     output_path= output_path + "/mha2nnunet_settings.json"
# )

In [ ]:
# # Initialize the converter to match images and labels for 3D nnU-net

# archive = MHA2nnUNetConverter(
#     scans_dir=images_path,
#     annotations_dir=annotations_path,
#     output_dir = cnn_output_path + "/cnn_raw_data",
#     mha2nnunet_settings=cnn_output_path +"/mha2nnunet_settings.json"
#      )

# # This command automates the pairing process
# archive.convert()
# archive.create_dataset_json()

In [ ]:
# # Initialize the converter to match images and labels for 3D nnU-net
# archive = MHA2nnUNetConverter(
#     scans_dir=images_path,
#     annotations_dir=annotations_path,
#     output_dir = output_path + "/nnUNet_raw_data",
#     mha2nnunet_settings=output_path +"/mha2nnunet_settings.json"
#      )

# # This command automates the pairing process
# archive.convert()
# archive.create_dataset_json()

In [ ]:
# Path to a sample mha image
image_path = output_path +'/nnUNet_raw_data/Task2201_picai_baseline/imagesTr/10258_1000262_0002.nii.gz'
img = sitk.ReadImage(image_path)

print(f"Size: {img.GetSize()}")           # Dimension (x, y, z)
print(f"Spacing: {img.GetSpacing()}")     # Voxel size in mm
print(f"Origin: {img.GetOrigin()}")

In [ ]:
# def split_nnunet_data(base_dir, test_ratio=0.2, seed=42):
#     """
#     Splits data in imagesTr into imagesTs after conversion.
#     base_dir: Path to TaskXXX_Prostate folder
#     """
#     random.seed(seed)

#     img_tr_path = Path(base_dir) / "imagesTr"
#     lbr_tr_path = Path(base_dir) / "labelsTr"
#     img_ts_path = Path(base_dir) / "imagesTs"
#     lbl_ts_path = Path(base_dir) / "labelsTs"

#     # Create imagesTs if it doesn't exist, and create parent directories if they don't exist
#     img_ts_path.mkdir(parents=True, exist_ok=True)
#     lbl_ts_path.mkdir(parents=True, exist_ok=True)

#     # 1. Get unique Patient IDs
#     # PI-CAI naming: [PatientID]_[StudyID]_[Modality].nii.gz
#     # We split by the first part of the filename (the PatientID)
#     all_files = [f for f in os.listdir(img_tr_path) if f.endswith(".nii.gz")]
#     patient_ids = list(set([f.split('_')[0] for f in all_files]))

#     random.shuffle(patient_ids)

#     # 2. Determine split point
#     split_idx = int(len(patient_ids) * test_ratio)
#     test_patients = patient_ids[:split_idx]

#     print(f"Total Patients: {len(patient_ids)}")
#     print(f"Moving {len(test_patients)} patients to the test set.")

#     # 3. Move files
#     for pid in test_patients:
#         # Move all modalities for this patient (_0000, _0001, _0002)
#         patient_images = [f for f in os.listdir(img_tr_path) if f.startswith(pid)]
#         for img in patient_images:
#             shutil.move(img_tr_path / img, img_ts_path / img)

#         # Move corresponding label
#         # Labels are named [PatientID]_[StudyID].nii.gz (no modality suffix)
#         # We find all labels starting with this patient ID
#         patient_labels = [f for f in os.listdir(lbl_tr_path) if f.startswith(pid)]
#         for lbl in patient_labels:
#             # For inference, you might want to keep labels in a separate 'labelsTs' folder
#             # or a 'hold_out' folder, as nnU-Net's imagesTs is usually for unlabeled data.
#             shutil.move(lbl_tr_path / lbl, lbl_ts_path / lbl)

#     print("✅ Split complete.")

# if __name__ == "__main__":
#     # Correcting the TASK_DIR path to avoid duplication of 'nnU-net'
#     TASK_DIR = Path(output_path) / "nnUNet_raw_data" / "Task2201_picai_baseline"
#     split_nnunet_data(TASK_DIR, test_ratio=0.2)

In [ ]:
# #Check Image Sizes

# # Path to your PICAI .mha file
# mha_file_path = '/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/cnn/cnn_raw_data/Task2201_picai_baseline/imagesTr/11290_1001313_0002.nii.gz'

# # Read the image
# image = sitk.ReadImage(mha_file_path)

# # Convert image to a numpy array (3D)
# # PICAI images are usually (z, y, x) or (slices, height, width)
# img_array = sitk.GetArrayFromImage(image)

# print(f"Image shape: {img_array.shape}")
# print(f"Spacing: {image.GetSpacing()}")


In [ ]:
# # 1. Define the REAL base (the parent of nnUNet_raw_data)
# base_path = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net"

# # 2. Set the variables
# os.environ['nnUNet_raw_data_base'] = base_path
# os.environ['nnUNet_preprocessed'] = os.path.join(base_path, "nnUNet_preprocessed")
# os.environ['RESULTS_FOLDER'] = os.path.join(base_path, "nnUNet_trained_models")

# # 3. Verify the path exists before running
# task_path = os.path.join(base_path, "nnUNet_raw_data/Task2201_picai_baseline")
# if os.path.exists(task_path):
#     print(f"✅ Found Task 2201 at: {task_path}")
# else:
#     print(f"❌ Error: Task folder not found at {task_path}. Check your folder names!")

In [ ]:
# task_path = Path("/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline")
# images_tr = task_path / "imagesTr"
# labels_tr = task_path / "labelsTr"
# images_ts = task_path / "imagesTs"
# images_ts.mkdir(exist_ok=True)

# # Get all unique case IDs from images (stripping the _000X suffix)
# image_cases = set(["_".join(f.split('_')[:2]) for f in os.listdir(images_tr) if f.endswith(".nii.gz")])

# # Get all case IDs from labels
# label_cases = set([f.replace(".nii.gz", "") for f in os.listdir(labels_tr) if f.endswith(".nii.gz")])

# # Find cases in images but NOT in labels
# orphans = image_cases - label_cases

# if orphans:
#     print(f"Found {len(orphans)} orphaned cases. Moving to imagesTs...")
#     for case in orphans:
#         for f in images_tr.glob(f"{case}_*.nii.gz"):
#             shutil.move(str(f), str(images_ts / f.name))
#             print(f"Moved: {f.name}")
# else:
#     print("No orphaned images found! Your folders are synchronized.")

In [ ]:
# # Define paths
# task_path = Path("/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline")
# images_tr = task_path / "imagesTr"

# # Get all unique case IDs currently in imagesTr
# # (Stripping the _0000, _0001 suffixes)
# current_training_cases = sorted(list(set(["_".join(f.name.split('_')[:2]).replace(".nii.gz", "")
#                                          for f in images_tr.glob("*.nii.gz")])))

# # Load the existing JSON
# json_path = task_path / "dataset.json"
# with open(åjson_path, 'r') as f:
#     data = json.load(f)

# # Update the training list and the number of training cases
# data['training'] = [{"image": f"./imagesTr/{case}.nii.gz", "label": f"./labelsTr/{case}.nii.gz"}
#                     for case in current_training_cases]
# data['numTraining'] = len(current_training_cases)

# # Overwrite the old JSON
# with open(json_path, 'w') as f:
#     json.dump(data, f, indent=4)

# print(f"✅ Updated dataset.json with {len(current_training_cases)} cases.")

In [ ]:

label_dir = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/labelsTr"
expected_labels = [0, 1] # Based on your JSON {0: background, 1: csPCa}

print("Checking mask values...")
found_unexpected = False

for label_file in os.listdir(label_dir):
    if label_file.endswith(".nii.gz"):
        img = nib.load(os.path.join(label_dir, label_file))
        data = img.get_fdata()
        unique_values = np.unique(data)

        # Check if values are in our expected list
        for val in unique_values:
            if val not in expected_labels:
                print(f"❌ Error in {label_file}: Found value {int(val)} which is NOT in dataset.json")
                found_unexpected = True

if not found_unexpected:
    print("✅ All masks contain only allowed values (0 and 1).")

In [ ]:
# Path to your labels
label_dir = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/labelsTr"

print("Starting label remapping...")

for label_file in os.listdir(label_dir):
    if label_file.endswith(".nii.gz"):
        file_path = os.path.join(label_dir, label_file)

        # Load the image
        img = nib.load(file_path)
        data = img.get_fdata()

        # Check if there are values other than 0 and 1
        unique_vals = np.unique(data)
        if not np.all(np.isin(unique_vals, [0, 1])):
            print(f"Fixing {label_file}: Found values {unique_vals}")

            # REMAP: Anything greater than 0 becomes 1
            # (Adjust this logic if 3 should actually be 0)
            data[data > 0] = 1

            # Create a new NIfTI image with the corrected data
            # We use the same affine and header to keep it aligned
            new_img = nib.Nifti1Image(data.astype(np.uint8), img.affine, img.header)
            nib.save(new_img, file_path)
            print(f"✅ {label_file} has been corrected.")

print("\nProcessing complete. All masks are now strictly 0 and 1.")

In [ ]:
task_path = Path("/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline")
img_dir = task_path / "imagesTr"
lbl_dir = task_path / "labelsTr"

# 1. Get exact list of files present in BOTH folders
images = {f.name.split('_0000')[0].split('_0001')[0].split('_0002')[0].replace(".nii.gz", "")
          for f in img_dir.glob("*.nii.gz")}
labels = {f.name.replace(".nii.gz", "") for f in lbl_dir.glob("*.nii.gz")}

# Intersection ensures only cases with BOTH image and label are included
valid_cases = sorted(list(images.intersection(labels)))

# 2. Build the JSON structure from scratch
dataset_info = {
    "name": "Task2201_picai_baseline",
    "description": "Prostate cancer detection",
    "reference": "PI-CAI",
    "licence": "CC-BY-NC-SA",
    "relpath": ".",
    "tensorImageSize": "4D",
    "modality": {
        "0": "T2W",
        "1": "ADC",
        "2": "HBV"
    },
    "labels": {
        "0": "background",
        "1": "csPCa"
    },
    "numTraining": len(valid_cases),
    "numTest": len(list((task_path / "imagesTs").glob("*.nii.gz"))),
    "training": [{"image": f"./imagesTr/{c}.nii.gz", "label": f"./labelsTr/{c}.nii.gz"} for c in valid_cases],
    "test": [] # nnU-Net handles test set discovery automatically
}

with open(task_path / "dataset.json", 'w') as f:
    json.dump(dataset_info, f, indent=4)

print(f"✅ Hard reset complete. dataset.json now has exactly {len(valid_cases)} valid cases.")

In [ ]:
# # Delete all files containing " (1)" in the imagesTr folder
# !find "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/imagesTr/" -name "* (1)*" -delete

# # Delete all files containing " (1)" in the labelsTr folder (just in case)
# !find "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/labelsTr/" -name "* (1)*" -delete

# print("✅ Cleanup complete. Duplicate 'copy' files have been removed.")

In [ ]:
!nnUNet_plan_and_preprocess -t 2201 --verify_dataset_integrity


#### nnU-net Model Training

In [ ]:
# 1. Setup Paths
base_dir = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net"
img_tr_path = Path(base_dir) / "nnUNet_raw_data/Task2201_picai_baseline/imagesTr"
output_path = Path(base_dir) / "nnUNet_preprocessed/Task2201_picai_baseline/splits_final.json"

# 2. Identify cases actually present in the folder
# We strip the modality suffix (_0000) to get the base Case ID
available_cases = set(["_".join(f.name.split('_')[:2]).replace(".nii.gz", "")
                      for f in img_tr_path.glob("*.nii.gz")])

print(f"Total cases found in imagesTr: {len(available_cases)}")

# 3. Filter official splits based on what we actually have
new_splits = []
all_split_cases = set()

for fold in range(5):
    # Filter official train/val lists to only include available files
    filt_train = [c for c in train_splits[fold]['subject_list'] if c in available_cases]
    filt_val = [c for c in valid_splits[fold]['subject_list'] if c in available_cases]

    new_splits.append({
        "train": filt_train,
        "val": filt_val
    })

    all_split_cases.update(filt_train)
    all_split_cases.update(filt_val)
    print(f"Fold {fold}: {len(filt_train)} train, {len(filt_val)} val")

# 4. Handle "Orphan" cases (files in folder but NOT in official PI-CAI splits)
orphans = available_cases - all_split_cases
if orphans:
    print(f"⚠️ Found {len(orphans)} cases in folder not in official splits. Adding to Fold 0 training.")
    new_splits[0]['train'].extend(list(orphans))

# 5. Save the synchronized JSON
with open(output_path, 'w') as f:
    json.dump(new_splits, f, indent=4)

print(f"\n✅ Synchronized splits_final.json saved to: {output_path}")

In [ ]:

# 1. Setup paths
base_dir = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net"
preprocessed_dir = Path(base_dir) / "nnUNet_preprocessed/Task2201_picai_baseline"
json_split_path = preprocessed_dir / "splits_final.json"
pkl_split_path = preprocessed_dir / "splits_final.pkl"

# 2. Identify the EXACT keys nnU-Net expects
# These are the names of the .npz files without the extension
dataset_keys = sorted([f.replace(".npz", "") for f in os.listdir(preprocessed_dir / "nnUNetData_plans_v2.1_stage0") if f.endswith(".npz")])

print(f"nnU-Net sees {len(dataset_keys)} cases in the preprocessed folder.")

# 3. Load your JSON splits and FILTER them to match the exact keys above
with open(json_split_path, 'r') as f:
    json_splits = json.load(f)

final_splits = []
for fold_data in json_splits:
    # We only keep IDs that actually exist in our preprocessed data
    train_keys = [k for k in fold_data['train'] if k in dataset_keys]
    val_keys = [k for k in fold_data['val'] if k in dataset_keys]

    final_splits.append({
        'train': train_keys,
        'val': val_keys
    })

# 4. Save as .pkl (This is what nnU-Net really wants)
with open(pkl_split_path, 'wb') as f:
    pickle.dump(final_splits, f)

print(f"✅ Created splits_final.pkl with {len(dataset_keys)} synchronized cases.")

In [ ]:
# 1. Create a local directory in the Colab instance
local_preprocessed = "/content/nnUNet_preprocessed"
os.makedirs(local_preprocessed, exist_ok=True)

# 2. Copy the preprocessed data from Drive to Local (this takes 5-10 mins)
# Replace the path below with your actual Drive path
drive_path = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_preprocessed/Task2201_picai_baseline"
local_task_path = os.path.join(local_preprocessed, "Task2201_picai_baseline")

print("🚀 Copying data to local disk for faster I/O...")
shutil.copytree(drive_path, local_task_path)

# 3. Update your Environment Variable to point to LOCAL
os.environ['nnUNet_preprocessed'] = local_preprocessed

In [ ]:
# 1. Find the path to the trainer file
nnunet_path = nnunet.__path__[0]
trainer_file = os.path.join(nnunet_path, "training/network_training/nnUNetTrainerV2.py")

print(f"Targeting: {trainer_file}")

# 2. Use 'sed' to replace 1000 with 250
# We use ! to run a shell command from within Python
!sed -i 's/self.max_num_epochs = 1000/self.max_num_epochs = 100/g' "{trainer_file}"

# 3. Verify the change
print("\nVerification:")
!grep "self.max_num_epochs =" "{trainer_file}"

In [ ]:
start_time = time.time()

!nnUNet_train 3d_fullres nnUNetTrainerV2 2201 0

end_time = time.time()
print("Training Time", end_time - start_time)


In [ ]:
# 1. Find the path to the trainer file
nnunet_path = nnunet.__path__[0]
trainer_file = os.path.join(nnunet_path, "training/network_training/nnUNetTrainerV2.py")

print(f"Targeting: {trainer_file}")

# 2. Use 'sed' to replace 1000 with 250
# We use ! to run a shell command from within Python
!sed -i 's/self.max_num_epochs = 100/self.max_num_epochs = 10/g' "{trainer_file}"

# 3. Verify the change
print("\nVerification:")
!grep "self.max_num_epochs =" "{trainer_file}"

In [ ]:
start_time = time.time()

!nnUNet_train 3d_fullres nnUNetTrainerV2 2201 1

end_time = time.time()
print("Training Time", end_time - start_time)

In [ ]:
start_time = time.time()

!nnUNet_train 3d_fullres nnUNetTrainerV2 2201 2

end_time = time.time()
print("Training Time", end_time - start_time)

In [ ]:
start_time = time.time()

!nnUNet_train 3d_fullres nnUNetTrainerV2 2201 3

end_time = time.time()
print("Training Time", end_time - start_time)

In [ ]:
start_time = time.time()

!nnUNet_train 3d_fullres nnUNetTrainerV2 2201 4

end_time = time.time()
print("Training Time", end_time - start_time)

In [ ]:
## Find where the 'engine' (PyTorch) loading logic is hidden in nnU-Net
nnunet_path = nnunet.__path__[0]
target_file = os.path.join(nnunet_path, "training/model_restore.py")

print(f"Applying safety patch to: {target_file}")

# This command replaces the strict 'load' with a 'load(..., weights_only=False)'
!sed -i "s/torch.load(i, map_location=torch.device('cpu'))/torch.load(i, map_location=torch.device('cpu'), weights_only=False)/g" "{target_file}"
!sed -i "s/torch.load(checkpoint_file, map_location=torch.device('cpu'))/torch.load(checkpoint_file, map_location=torch.device('cpu'), weights_only=False)/g" "{target_file}"

print("\n✅ Patch applied. The 'engine' will now let nnU-Net load your weights.")

In [ ]:
!nnUNet_predict -i "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/imagesTs" \
                -o "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/predictions" \
                -t 2201 \
                -m 3d_fullres
                # --save_npz

In [ ]:


# Update these paths to your actual folders
img_ts_path = Path("/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/imagesTs")
pred_path = Path("/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/predictions")

# 1. Get all unique Case IDs from the input folder
# (Stripping the _000X.nii.gz suffix)
input_cases = set(["_".join(f.name.split('_')[:2]).replace(".nii.gz", "")
                  for f in img_ts_path.glob("*.nii.gz")])

# 2. Get all unique Case IDs from the prediction folder
predicted_cases = set([f.name.replace(".nii.gz", "")
                      for f in pred_path.glob("*.nii.gz")])

missing_cases = input_cases - predicted_cases

print(f"Total input cases found: {len(input_cases)}")
print(f"Total predictions found: {len(predicted_cases)}")
print(f"Missing predictions: {len(missing_cases)}")

if missing_cases:
    print("\n--- Diagnostic Check for Missing Cases ---")
    for case in sorted(list(missing_cases))[:5]: # Show first 5
        print(f"\nChecking Case: {case}")
        for mod in ["_0000", "_0001", "_0002"]:
            exists = (img_ts_path / f"{case}{mod}.nii.gz").exists()
            status = "✅ Found" if exists else "❌ MISSING"
            print(f"  Modality {mod}: {status}")

In [ ]:
# -ref: Path to your Ground Truth (labelsTs)
# -pred: Path to your Predicted Masks (predictions)
# -l: The label ID to evaluate (1 for csPCa)

!nnUNet_evaluate_folder -ref  "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/labelsTs" \
                        -pred "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/predictions" \
                        -l 1

In [ ]:
def visual_check(case_id, img_dir, pred_dir):
    # 1. Load the Image (T2W) and the Prediction
    img_path = os.path.join(img_dir, f"{case_id}_0000.nii.gz")
    pred_path = os.path.join(pred_dir, f"{case_id}.nii.gz")

    img_data = nib.load(img_path).get_fdata()
    pred_data = nib.load(pred_path).get_fdata()

    # 2. Find the slice with the most predicted cancer pixels
    # We sum the mask along the X and Y axes to find the Z-slice with max intensity
    z_sums = np.sum(pred_data, axis=(0, 1))
    z_slice = np.argmax(z_sums) if np.max(z_sums) > 0 else img_data.shape[2] // 2

    # 3. Plotting
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Slice of Raw MRI
    axes[0].imshow(img_data[:, :, z_slice].T, cmap='gray', origin='lower')
    axes[0].set_title(f"Raw T2W (Slice {z_slice})")
    axes[0].axis('off')

    # Slice of Predicted Mask
    axes[1].imshow(pred_data[:, :, z_slice].T, cmap='viridis', origin='lower')
    axes[1].set_title("Predicted Mask Only")
    axes[1].axis('off')

    # The Overlay
    axes[2].imshow(img_data[:, :, z_slice].T, cmap='gray', origin='lower')
    axes[2].imshow(pred_data[:, :, z_slice].T, cmap='Reds', alpha=0.5, origin='lower')
    axes[2].set_title("Overlay (Detection)")
    axes[2].axis('off')

    plt.suptitle(f"Visual Check for Case: {case_id}", fontsize=16)
    plt.show()

# --- RUN THE CHECK ---
# Replace these with your actual paths
IMAGE_DIR = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/imagesTs"
PRED_DIR = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/predictions"

# Pick a case ID from your predictions folder
example_case = "10004_1000004"
visual_check(example_case, IMAGE_DIR, PRED_DIR)

In [ ]:
metrics_obj = evaluate_folder(
    y_det_dir="/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/predictions",
    y_true_dir= "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/labelsTs",
    subject_list=None # It will automatically match filenames
)

# The aggregated metrics are directly available as attributes of the Metrics object

print(f"Overall PI-CAI Score (AUROC): {metrics_obj.auroc}")
print(f"AUROC: {metrics_obj.auroc}")
print(f"AP: {metrics_obj.AP}")

In [ ]:
# --- CONFIGURATION ---
PRED_DIR = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/predictions"
LABELS_DIR = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/labelsTs"
SUMMARY_JSON = f"{PRED_DIR}/summary.json"

# 1. Get Dice from nnU-Net summary
with open(SUMMARY_JSON, 'r') as f:
    data = json.load(f)
    # Assuming '1' is the foreground class and exists in the summary
    dice_score = data['results']['mean']['1']['Dice'] if '1' in data['results']['mean'] else np.nan

# 2. Get AUROC and AP from picai_eval
metrics = evaluate_folder(
    y_det_dir=PRED_DIR,
    y_true_dir=LABELS_DIR
    # return_case_metrics=True # This argument caused a TypeError and has been removed.
)
auroc = metrics.auroc
ap = metrics.AP

# 3. Calculate Patient-level Sensitivity and Specificity (currently disabled due to API error)
# tp, fp, tn, fn = 0, 0, 0, 0
# for case_id in metrics.subject_list:
#     # Access per-case metrics for class '1' (csPCa lesion)
#     case_metrics_for_class_1 = metrics.cases[case_id]['1']

#     # Check if the ground truth for this case has any positive lesions
#     has_gt = case_metrics_for_class_1['Total Positives Reference'] > 0

#     # Check if the prediction for this case has any positive lesions
#     has_pred = case_metrics_for_class_1['Total Positives Test'] > 0

#     if has_gt and has_pred: tp += 1
#     elif not has_gt and has_pred: fp += 1
#     elif not has_gt and not has_pred: tn += 1
#     elif has_gt and not has_pred: fn += 1

# sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
# specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

sensitivity = np.nan # Set to NaN as calculation is disabled
specificity = np.nan # Set to NaN as calculation is disabled

# 4. Generate the Table
results = {
    "Metric": ["Dice Similarity (DSC)", "Lesion-level AUC", "Average Precision (AP)", "Sensitivity", "Specificity"],
    "Score": [dice_score, auroc, ap, sensitivity, specificity]
}
df = pd.DataFrame(results)
print(df.round(4))


In [ ]:
# --- Dummy Data Setup (Replace these with your actual flat arrays) ---
# y_true: Ground truth labels (0 or 1)
# y_scores: Predicted probabilities (0.0 to 1.0)
# Example: y_true = labels.flatten(), y_scores = probabilities.flatten()

# Dummy data for demonstration
np.random.seed(42)
all_labels = np.random.randint(0, 2, 100) # 100 random binary labels
all_probs = np.random.rand(100) # 100 random probabilities

def plot_roc_curve(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') # Random guess line

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])

    # Sensitivity = TPR; Specificity = 1 - FPR
    plt.xlabel('1 - Specificity (False Positive Rate)')
    plt.ylabel('Sensitivity (True Positive Rate)')
    plt.title('Receiver Operating Characteristic (ROC) - Fold 0')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

plot_roc_curve(all_labels, all_probs)

In [ ]:
def plot_sens_vs_spec(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    specificity = 1 - fpr

    plt.figure(figsize=(8, 6))
    plt.plot(specificity, tpr, color='blue', lw=2, label='Sensitivity-Specificity Trade-off')

    plt.xlabel('Specificity')
    plt.ylabel('Sensitivity')
    plt.title('Sensitivity vs. Specificity Curve')
    plt.gca().invert_xaxis() # Specificity usually goes 1.0 -> 0.0
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
def find_optimal_threshold(y_true, y_probs):
    # 1. Calculate the ROC curve points
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)

    # 2. Calculate Youden's J statistic for every point on the curve
    # J = Sensitivity + Specificity - 1
    # Note: Specificity = 1 - FPR, so J = TPR - FPR
    j_scores = tpr - fpr

    # 3. Find the index of the maximum J score
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]

    # Metrics at that specific threshold
    opt_sensitivity = tpr[optimal_idx]
    opt_specificity = 1 - fpr[optimal_idx]

    return optimal_threshold, opt_sensitivity, opt_specificity, fpr, tpr

# --- EXECUTION ---
# Replace 'all_labels' and 'all_probs' with your flattened test data
threshold, sens, spec, fpr, tpr = find_optimal_threshold(all_labels, all_probs)

print(f"🎯 Optimal Threshold (Youden's J): {threshold:.4f}")
print(f"📊 Sensitivity at this point: {sens:.4f}")
print(f"📊 Specificity at this point: {spec:.4f}")

In [ ]:
def plot_roc_with_elbow(fpr, tpr, sens, spec, threshold):
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC Curve')
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')

    # Mark the optimal point (FPR = 1 - Specificity)
    plt.scatter(1 - spec, sens, color='red', s=100, label=f'Optimal Point (T={threshold:.2f})', zorder=5)

    plt.xlabel('1 - Specificity')
    plt.ylabel('Sensitivity')
    plt.title('ROC Curve with Optimal Threshold (Youden\'s J)')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

plot_roc_with_elbow(fpr, tpr, sens, spec, threshold)

### Working with Transfer Learning Models

##### Splitting the Dataset

In [ ]:
# --- CONFIGURATION ---
# Use your 'imagesTs' (Test set) for final evaluation
mask_tr_dir="/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/labelsTr"
img_tr_dir="/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/imagesTr"
img_ts_dir = "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/imagesTs"
mask_pr_dir= "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/predictions" # The output from your nnU-Net
mask_ts_dir= "/content/drive/My Drive/Colab Notebooks/DBA/MSC Capstone Project/output/nnU-net/nnUNet_raw_data/Task2201_picai_baseline/labelsTs"


In [ ]:
def get_bounding_box(mask, padding=5):
    """
    Finds the min and max indices of non-zero pixels and adds padding.
    Assumes mask is (X, Y, Z) for 3D data from nibabel.get_fdata().
    """
    coords = np.argwhere(mask > 0)
    if coords.size == 0:
        return None  # No lesion found

    # Get min/max for each dimension. coords will be (X_coord, Y_coord, Z_coord)
    x_min_coord, y_min_coord, z_min_coord = coords.min(axis=0)
    x_max_coord, y_max_coord, z_max_coord = coords.max(axis=0)

    # Add padding and handle boundaries
    x_min = max(0, x_min_coord - padding)
    y_min = max(0, y_min_coord - padding)
    z_min = max(0, z_min_coord - padding)

    x_max = min(mask.shape[0], x_max_coord + padding)
    y_max = min(mask.shape[1], y_max_coord + padding)
    z_max = min(mask.shape[2], z_max_coord + padding)

    return (x_min, x_max, y_min, y_max, z_min, z_max)

In [ ]:
def create_input(case_id, img_dir, mask_dir, target_size=(64, 64, 64)):
    # 1. Load the Mask to determine if it's a positive case or for ROI
    mask_path = f"{mask_dir}/{case_id}.nii.gz"
    mask_obj = nib.load(mask_path)
    mask_data = mask_obj.get_fdata() # (X, Y, Z) order from nibabel

    bbox = get_bounding_box(mask_data) # This returns (x_min, x_max, y_min, y_max, z_min, z_max) or None

    channels = []
    # Loop through modalities
    for mod in ["_0000", "_0001", "_0002"]:
        img_path = f"{img_dir}/{case_id}{mod}.nii.gz"
        img_data = nib.load(img_path).get_fdata() # (X, Y, Z) order from nibabel

        if bbox is None:
            # Healthy case: take a central crop
            current_x, current_y, current_z = img_data.shape

            # Define central crop dimensions for healthy images.
            # Take a central crop of 96x96 in X/Y (or larger if image is large), keeping full Z-dimension,
            # then it will be scaled to target_size (64x64x64).
            central_crop_xy_size = 96 # A common practice, adjust as needed

            x_center = current_x // 2
            y_center = current_y // 2

            x_start_healthy = max(0, x_center - central_crop_xy_size // 2)
            x_end_healthy = min(current_x, x_center + central_crop_xy_size // 2)
            y_start_healthy = max(0, y_center - central_crop_xy_size // 2)
            y_end_healthy = min(current_y, y_center + central_crop_xy_size // 2)

            # Extract crop
            crop_healthy = img_data[x_start_healthy:x_end_healthy, y_start_healthy:y_end_healthy, :]

            # Pad if the cropped region is smaller than desired central_crop_xy_size
            # This can happen if the original image itself is very small
            pad_x_before = (central_crop_xy_size - crop_healthy.shape[0]) // 2
            pad_x_after = central_crop_xy_size - crop_healthy.shape[0] - pad_x_before
            pad_y_before = (central_crop_xy_size - crop_healthy.shape[1]) // 2
            pad_y_after = central_crop_xy_size - crop_healthy.shape[1] - pad_y_before

            crop_healthy = np.pad(crop_healthy,
                                  ((pad_x_before, pad_x_after),
                                   (pad_y_before, pad_y_after),
                                   (0,0)), mode='constant', constant_values=0) # Pad with zeros

            crop = crop_healthy
        else:
            # Lesion case: crop around bounding box
            x_min_bbox, x_max_bbox, y_min_bbox, y_max_bbox, z_min_bbox, z_max_bbox = bbox
            crop = img_data[x_min_bbox:x_max_bbox, y_min_bbox:y_max_bbox, z_min_bbox:z_max_bbox]

        # 3. Resample to target size (target_X, target_Y, target_Z)
        # crop.shape is (current_X_dim, current_Y_dim, current_Z_dim)
        # target_size is (target_X_dim, target_Y_dim, target_Z_dim)
        factors = [t / c for t, c in zip(target_size, crop.shape)]
        crop_resized = zoom(crop, factors, order=1) # Linear interpolation

        # 4. Simple Z-Score Normalization
        crop_normalized = (crop_resized - np.mean(crop_resized)) / (np.std(crop_resized) + 1e-8)
        channels.append(crop_normalized)

    # Stack to create a (3, target_X_dim, target_Y_dim, target_Z_dim) tensor (channels, depth, height, width)
    return np.stack(channels, axis=0)

In [ ]:
output_train_dir = "/content/output_train_data" # Use local disk for speed!
os.makedirs(output_train_dir, exist_ok=True)

# Lists to hold our data and corresponding case_ids and labels
processed_data = [] # Will store (cube_tf, label, case_id)

# Get unique Case IDs
case_ids = sorted(list(set([os.path.basename(f).split('_')[0] + '_' + os.path.basename(f).split('_')[1] for f in os.listdir(img_tr_dir) if f.endswith(".nii.gz")])))

for case in tqdm(case_ids):
    mask_path = os.path.join(mask_tr_dir, f"{case}.nii.gz")
    if not os.path.exists(mask_path):
        # Skip if mask file does not exist for some reason
        continue

    mask_data = nib.load(mask_path).get_fdata()
    is_positive = np.max(mask_data) > 0
    label = 1 if is_positive else 0

    cube = create_input(case, img_tr_dir, mask_tr_dir, target_size=(64, 64, 64))

    if cube is not None:
        cube_tf = np.transpose(cube, (1, 2, 3, 0))
        processed_data.append((cube_tf, label, case))

# Separate the components into global variables
# global all_images_processed, all_labels_processed, all_case_ids_processed
all_images_processed = np.array([item[0] for item in processed_data])
all_labels_processed = np.array([item[1] for item in processed_data])
all_case_ids_processed = np.array([item[2] for item in processed_data])

print(f"\n✅ Extraction Complete!")
print(f"Total processed cubes: {len(processed_data)}")
print(f"Positive Cubes (Cancer): {np.sum(all_labels_processed == 1)}")
print(f"Negative Cubes (Healthy): {np.sum(all_labels_processed == 0)}")

In [ ]:
output_test_dir = "/content/output_test_data" # Use local disk for speed!
os.makedirs(output_test_dir, exist_ok=True)

# Lists to hold our data and corresponding case_ids and labels
test_processed_data = [] # Will store (cube_tf, label, case_id)

# Get unique Case IDs
case_ids = sorted(list(set([os.path.basename(f).split('_')[0] + '_' + os.path.basename(f).split('_')[1] for f in os.listdir(img_ts_dir) if f.endswith(".nii.gz")]))) # Corrected: Use img_ts_dir

for case in tqdm(case_ids):
    mask_path = os.path.join(mask_ts_dir, f"{case}.nii.gz") # Corrected: Use mask_ts_dir
    if not os.path.exists(mask_path):
        # Skip if mask file does not exist for some reason
        continue

    mask_data = nib.load(mask_path).get_fdata()
    is_positive = np.max(mask_data) > 0
    label = 1 if is_positive else 0

    cube = create_input(case, img_ts_dir, mask_ts_dir, target_size=(64, 64, 64))

    if cube is not None:
        cube_tf = np.transpose(cube, (1, 2, 3, 0))
        test_processed_data.append((cube_tf, label, case))

# Separate the components into global variables
global all_test_images_processed, all_test_labels_processed, all_test_case_ids_processed
all_test_images_processed = np.array([item[0] for item in test_processed_data])
all_test_labels_processed = np.array([item[1] for item in test_processed_data])
all_test_case_ids_processed = np.array([item[2] for item in test_processed_data])

print(f"\n✅ Extraction Complete!")
print(f"Total processed cubes: {len(test_processed_data)}")
print(f"Positive Cubes (Cancer): {np.sum(all_test_labels_processed == 1)}")
print(f"Negative Cubes (Healthy): {np.sum(all_test_labels_processed == 0)}")

In [ ]:
# Assume df is available from earlier execution
# all_images_processed, all_labels_processed, all_case_ids_processed come from the previous cell

# Split image data, labels, and case IDs together to maintain alignment
X_train_img, X_val_img, y_train, y_val, \
X_train_case_ids, X_val_case_ids = train_test_split(
    all_images_processed, all_labels_processed, all_case_ids_processed,
    test_size=0.2, random_state=42, stratify=all_labels_processed
)

# Assign the test Images directly as no further splitting is desired for the test set
X_test_img = all_test_images_processed
y_test = all_test_labels_processed
X_test_case_ids = all_test_case_ids_processed


print(f"Total Images for Training: {len(X_train_img)}")
print(f"Total Images for Testing: {len(X_test_img)}")
print(f"Total Images for Validation: {len(X_val_img)}")

# --- Align Clinical Data ---

# 1. Ensure df has a 'case_id' column that matches 'patient_id_study_id' format
# This is crucial for matching clinical data with image data.
if 'case_id' not in df.columns:
    df['case_id'] = df['patient_id'].astype(str) + '_' + df['study_id'].astype(str)

# Create a common DataFrame for clinical data that we can filter and sort
# Use all case IDs from both training and test image datasets for comprehensive clinical data.
all_unique_image_case_ids = np.unique(np.concatenate((all_case_ids_processed, all_test_case_ids_processed)))
df_clinical_filtered = df[df['case_id'].isin(all_unique_image_case_ids)].copy()

# Handle potential duplicates in df_clinical_filtered for the same case_id
df_clinical_filtered = df_clinical_filtered.drop_duplicates(subset=['case_id'])

# Define clinical features to be used
clinical_cols = ['patient_age', 'psa', 'psad','prostate_volume'] # Ensure these columns exist in df

# Create the FeatureUnion for imputation and missing indicators
transformer = FeatureUnion(
    transformer_list=[
        ('imputer', SimpleImputer(strategy='median')),
        ('indicator', MissingIndicator(sparse=False, features='all')) # Set features='all' to always generate indicator columns for all features
    ]
)

# Fit and transform the clinical data for all cases that are present in the image data
# We fit the transformer on the training portion to prevent data leakage
# First, prepare the clinical data for fitting the transformer
clinical_data_for_fit = df_clinical_filtered[df_clinical_filtered['case_id'].isin(X_train_case_ids)][clinical_cols]

transformer.fit(clinical_data_for_fit)

# Now, transform all filtered clinical data
transformed_clinical_data = transformer.transform(df_clinical_filtered[clinical_cols])

# Generate new column names for the transformed data
indicator_col_names = [f'{col}_is_missing' for col in clinical_cols]
new_clinical_col_names = clinical_cols + indicator_col_names

transformed_df = pd.DataFrame(transformed_clinical_data, columns=new_clinical_col_names, index=df_clinical_filtered.index)

# Drop original clinical columns and join with the transformed data in df_clinical_filtered
df_clinical_processed = df_clinical_filtered.drop(columns=clinical_cols).join(transformed_df)

# Scale the clinical features using StandardScaler trained on the training data
scaler = StandardScaler()

# Fit scaler only on training clinical data
X_train_clinical_features = df_clinical_processed[df_clinical_processed['case_id'].isin(X_train_case_ids)][new_clinical_col_names]
scaler.fit(X_train_clinical_features)

# Transform all splits
# Create a helper function to reorder clinical data based on image case IDs
def get_ordered_clinical_data(case_ids_order, processed_clinical_df, feature_cols, scaler):
    # Filter the processed clinical DataFrame to include only the relevant case_ids
    ordered_df = processed_clinical_df[processed_clinical_df['case_id'].isin(case_ids_order)].set_index('case_id').loc[case_ids_order]
    # Extract and scale the features
    scaled_features = scaler.transform(ordered_df[feature_cols])
    return scaled_features

X_train_clinical = get_ordered_clinical_data(X_train_case_ids, df_clinical_processed, new_clinical_col_names, scaler)
X_val_clinical = get_ordered_clinical_data(X_val_case_ids, df_clinical_processed, new_clinical_col_names, scaler)
X_test_clinical = get_ordered_clinical_data(X_test_case_ids, df_clinical_processed, new_clinical_col_names, scaler)

# Rename X_train_img to X_train and X_val_img to X_val for consistency with later cells
X_train = X_train_img
X_val = X_val_img
X_test = X_test_img # Make sure X_test is also updated if needed elsewhere

# Re-create pos_images and neg_images from X_train and y_train for balanced_ds
pos_images_train = X_train[y_train.flatten() == 1]
neg_images_train = X_train[y_train.flatten() == 0]
pos_labels_train = y_train[y_train.flatten() == 1]
neg_labels_train = y_train[y_train.flatten() == 0]

# Add checks for alignment:
print(f"Shape of X_train_clinical: {X_train_clinical.shape}")
print(f"Shape of X_val_clinical: {X_val_clinical.shape}")
print(f"Shape of X_test_clinical: {X_test_clinical.shape}")

print(f"Length of X_train_case_ids: {len(X_train_case_ids)}")
print(f"Length of X_val_case_ids: {len(X_val_case_ids)}")
print(f"Length of X_test_case_ids: {len(X_test_case_ids)}")

print(f"Number of clinical cases in df_clinical_processed for train: {len(df_clinical_processed[df_clinical_processed['case_id'].isin(X_train_case_ids)])}")
print(f"Number of clinical cases in df_clinical_processed for val: {len(df_clinical_processed[df_clinical_processed['case_id'].isin(X_val_case_ids)])}")
print(f"Number of clinical cases in df_clinical_processed for test: {len(df_clinical_processed[df_clinical_processed['case_id'].isin(X_test_case_ids)])}")


In [ ]:
X_train_clinical.shape

In [ ]:
df_clinical_processed.head(5)

In [ ]:
df_clinical_processed.head(5)

In [ ]:
df_clinical_processed.isnull().sum()

In [ ]:
def augment_3d(image, label):
    """Applies basic 3D augmentations to the MRI cube."""
    # image shape: (depth, height, width, channels)

    # Random Horizontal Flip (flip along width axis, axis=2)
    if tf.random.uniform([]) > 0.5:
        image = tf.reverse(image, axis=[2])

    # Random Vertical Flip (flip along height axis, axis=1)
    if tf.random.uniform([]) > 0.5:
        image = tf.reverse(image, axis=[1])

    # Cast image to float32 to match noise dtype
    image = tf.cast(image, dtype=tf.float32)

    # Add minor Gaussian noise to simulate MRI artifacts
    noise = tf.random.normal(shape=tf.shape(image), mean=0.0, stddev=0.05, dtype=tf.float32)
    image = image + noise

    return image, label

In [ ]:
# Define BATCH_SIZE
BATCH_SIZE = 8

# Create two separate data streams using the *training* data
pos_ds = tf.data.Dataset.from_tensor_slices((pos_images_train, pos_labels_train)).repeat()
neg_ds = tf.data.Dataset.from_tensor_slices((neg_images_train, neg_labels_train)).repeat()

# Interleave them so the model never sees a 'purely healthy' batch
train_ds = tf.data.Dataset.sample_from_datasets(
    [pos_ds, neg_ds],
    weights=[0.5, 0.5] # Equal probability of picking a cancer vs healthy cube
)

# Apply Augmentation and Optimization (to unbatched samples)
balanced_train_ds = train_ds.map(augment_3d, num_parallel_calls=tf.data.AUTOTUNE)
balanced_train_ds = balanced_train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Also create datasets for validation and test, for consistency
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


#### Treating Outliers with the Numpy Clip Function

In [ ]:
outliers_list=['patient_age','psa','psad','prostate_volume'] #list of columns, outliers are treated.
LWR=[]
UWR=[]
for i in outliers_list:
    Q1=df_clinical_processed[i].quantile(0.25) # 25th quantile
    Q3=df_clinical_processed[i].quantile(0.75)  # 75th quantile
    IQR=Q3-Q1
    LW = Q1 - 1.5*IQR   #LW--Low Whisker
    UW = Q3 + 1.5*IQR   #UW--Upper Whisker
    print (f'For column {i}: IQR={IQR},Low Whisker={LW},Upper Whisker={UW}')
    global LWR             # a list to hold the LW values
    global UWR             # a list to hold the UW values
    LWR.append(LW)
    UWR.append(UW)

In [ ]:
print ('UWR:','\n',UWR)
print ()
print ('LWR:','\n',LWR)

In [ ]:
# Confirming the outliers have been treated for each affected columns
# box_list1=['DurationOfPitch','NumberOfTrips']
plt.figure(figsize=(25,15))

for e,d in enumerate(outliers_list):
                     plt.subplot(3,4,e+1)
                     sns.boxplot(y=d,data=df_clinical_processed,whis=1.5)
                     plt.tight_layout()
                     plt.title(d)

In [ ]:
# Replacing the uppper and lower outliers with the upper and lower whisker values for each column

for z,y,x in zip(outliers_list,LWR,UWR):
    df_clinical_processed[z] = np.clip(df_clinical_processed[z], y, x)

In [ ]:
# Confirming the outliers have been treated for each affected columns
# box_list1=['DurationOfPitch','NumberOfTrips']
plt.figure(figsize=(25,15))

for e,d in enumerate(outliers_list):
                     plt.subplot(3,4,e+1)
                     sns.boxplot(y=d,data=df_clinical_processed,whis=1.5)
                     plt.tight_layout()
                     plt.title(d)

##### Model Performance Evaluation Metrics

In [ ]:
def plot_history(history):
    # Set a clean, academic style
    plt.style.use('seaborn-v0_8-muted')
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    # Helper to find keys dynamically
    def get_key(name):
        return [k for k in history.history.keys() if name in k.lower() and 'val' not in k.lower()][0]

    metrics_to_plot = [
        ('loss', 'Cross-Entropy Loss', 'Loss'),
        ('auc', 'Area Under ROC', 'AUC Score'),
        ('recall', 'Sensitivity (Recall)', 'Recall Score'),
        ('precision', 'Average Precision (AP)', 'AP Score'),
        ('specificity', 'Specificity', 'Specificity Score')

    ]

    for i, (key_search, title, ylabel) in enumerate(metrics_to_plot):
        try:
            train_key = get_key(key_search)
            val_key = f"val_{train_key}"
            axes[i].plot(history.history[train_key], label='Train', lw=2)
            axes[i].plot(history.history[val_key], label='Val', lw=2, linestyle='--')
            axes[i].set_title(title, fontsize=14, fontweight='bold')
            axes[i].set_xlabel('Epoch')
            axes[i].set_ylabel(ylabel)
            axes[i].legend()
            axes[i].grid(alpha=0.3)

        except IndexError:
            fig.delaxes(axes[i])
            # axes[i].set_title(f"{title} (Not Found)", color='red')

    # Remove the 6th empty subplot
    fig.delaxes(axes[5])


    plt.tight_layout()
    plt.show()

In [ ]:
# Helper function to compute Dice score
def compute_dice(y_true_mask, y_pred_mask):
    """Computes the Dice Similarity Coefficient for 3D volumes."""
    intersection = np.sum(y_true_mask * y_pred_mask)
    denominator = np.sum(y_true_mask) + np.sum(y_pred_mask)
    if denominator == 0: # Handle cases where both masks are empty
        return 1.0
    return (2. * intersection) / (denominator + 1e-8)

def inference_pipeline(model, X_images, X_case_ids, y_true_labels, name, threshold=0.34, clinical_data=None, multimodal=False):
    """
    Performs inference for a multimodal model on preprocessed image and clinical data
    and calculates classification and segmentation metrics for each case.

    Args:
        model (tf.keras.Model): The multimodal Keras model (e.g., res_mul_model).
        X_images (np.ndarray): Array of preprocessed image cubes (e.g., X_test).
        X_case_ids (np.ndarray): Array of case IDs corresponding to X_images and X_clinical.
        y_true_labels (np.ndarray): Array of true binary labels (0 or 1) for X_images.
        threshold (float): Decision threshold for binary classification.
        name (str): A name for the output CSV file.
        clinical_data (np.ndarray, optional): Array of preprocessed clinical features. Defaults to None.
        multimodal (bool): Flag indicating if the model is multimodal. Defaults to False.

    Returns:
        pd.DataFrame: A DataFrame containing Case_ID, Malignancy_Score, Final_Prediction, Actual_Label, and Dice_Score.
    """
    results = []

    print(f"🚀 Starting {'Multimodal' if multimodal else 'Single Input'} Inference on {len(X_case_ids)} cases for {name}...")

    # Ensure mask_ts_dir and mask_pr_dir are accessible (assumed global here or passed)
    global mask_ts_dir, mask_pr_dir

    for i in tqdm(range(len(X_case_ids))):
        case_id = X_case_ids[i]
        image_cube = X_images[i]
        clinical_vector = clinical_data[i] if multimodal and clinical_data is not None else None
        actual_label = y_true_labels[i]

        # Predict using the model
        img_input_batch = np.expand_dims(image_cube, axis=0) # Add batch dimension

        if multimodal:
            clin_input_batch = np.expand_dims(clinical_vector, axis=0) # Add batch dimension
            logits = model.predict([img_input_batch, clin_input_batch], verbose=0)
        else:
            logits = model.predict(img_input_batch, verbose=0)

        # Assuming model output is logits, convert to probability
        probability = tf.nn.sigmoid(logits).numpy()[0][0]
        prediction = 1 if probability > threshold else 0

        # --- Calculate Dice Score for the current case ---
        case_dice_score = np.nan # Default if masks are not found or cannot be processed

        gt_path = os.path.join(mask_ts_dir, f"{case_id}.nii.gz")
        pred_path = os.path.join(mask_pr_dir, f"{case_id}.nii.gz")

        if os.path.exists(gt_path) and os.path.exists(pred_path):
            try:
                gt_mask = nib.load(gt_path).get_fdata()
                pred_mask = nib.load(pred_path).get_fdata()
                case_dice_score = compute_dice(gt_mask, pred_mask)
            except Exception as e:
                print(f"Warning: Could not compute Dice for {case_id} due to {e}")
        else:
            # If either mask is missing, it's often a healthy case with no GT lesion
            # For malignant cases, a missing mask should probably result in a 0.0 Dice.
            # For healthy cases, a perfect Dice (no predicted lesion) is 1.0.
            if actual_label == 0:
                case_dice_score = 1.0
            else:
                case_dice_score = 0.0 # Malignant case but no masks found or predicted

        results.append({
            "Case_ID": case_id,
            "Malignancy_Score": probability,
            "Final_Prediction": prediction,
            "Actual_Label": actual_label,
            "Dice_Score": case_dice_score # Add Dice score
        })

    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{name}_inference.csv", index=False)
    print(f"\n✅ {'Multimodal' if multimodal else 'Single Input'} {name} Inference complete. Results saved to {name}_inference.csv")
    return results_df

In [ ]:
def plot_inference_distribution(results,model_name):
    plt.figure(figsize=(25, 15))
    sns.histplot(results['Malignancy_Score'], bins=20, kde=True, color='purple')
    plt.title(f'Distribution of {model_name} Malignancy Scores')
    plt.xlabel('Probability of Cancer')
    plt.ylabel('Number of Cases')
    plt.axvline(x=0.34, color='red', linestyle='--', label='Decision Threshold')
    plt.legend()
    plt.show()

    # 1. Load your results
    df = pd.read_csv(f'{model_name}_inference.csv')

    # 2. Get Ground Truth for each case in the CSV
    # We check if the mask in labelsTs contains any cancer (label 1)
    y_true = []
    print ('\n\n')
    print("Extracting ground truth labels...")
    for case_id in df['Case_ID']:
        label_path = os.path.join(mask_ts_dir, f"{case_id}.nii.gz")
        if os.path.exists(label_path):
            label_data = nib.load(label_path).get_fdata()
            y_true.append(1 if np.max(label_data) > 0 else 0)
        else:
            # If no label file exists, we assume healthy or skip
            y_true.append(0)

    df['Actual_Label'] = y_true

    print ('\n\n')

    # --- PLOT 1: CONFUSION MATRIX ---
    cm = confusion_matrix(df['Actual_Label'], df['Final_Prediction'])
    plt.figure(figsize=(25, 15))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Predicted Healthy', 'Predicted Cancer'],
                yticklabels=['Actual Healthy', 'Actual Cancer'])
    plt.title('Confusion Matrix: Two-Stage Pipeline (nnU-Net + ResNet)')
    plt.ylabel('Ground Truth')
    plt.xlabel('Pipeline Prediction')
    plt.show()

    # --- PLOT 2: PROBABILITY HISTOGRAM ---

    print ('\n\n')
    plt.figure(figsize=(25, 15))
    # Separate scores by actual class for a "stacked" or "side-by-side" look
    sns.histplot(data=df, x='Malignancy_Score', hue='Actual_Label',
                element="step", kde=True, palette={0: "blue", 1: "red"}, alpha=0.5, bins=20)

    # Add your threshold line
    threshold = 0.34
    plt.axvline(x=threshold, color='black', linestyle='--', label=f'Decision Threshold ({threshold})')

    plt.title('Distribution of ResNet Malignancy Scores by Actual Class')
    plt.xlabel('Probability of Malignancy')
    plt.ylabel('Count')
    plt.legend(title="Actual Class", labels=['Cancer (1)', 'Healthy (0)', 'Threshold'])
    plt.grid(axis='y', alpha=0.3)
    plt.show()

    # --- PRINT STATS ---
    print("\n--- Final Test Set Classification Report ---")
    print(classification_report(df['Actual_Label'], df['Final_Prediction'], target_names=['Healthy', 'Cancer']))

In [ ]:
def compute_dice(y_true_mask, y_pred_mask):
    """Computes the Dice Similarity Coefficient for 3D volumes."""
    intersection = np.sum(y_true_mask * y_pred_mask)
    return (2. * intersection) / (np.sum(y_true_mask) + np.sum(y_pred_mask) + 1e-8)


def inference_perf_metrics(model, X_test, y_test,clinical_data=None, multimodal=False):
    # --- 1. Run Inference ---
    # Predict probabilities (ensure your model was compiled with from_logits=True if needed)
    y_logits = model.predict([X_test, clinical_data]) if multimodal else model.predict(X_test)
    y_probs = tf.nn.sigmoid(y_logits).numpy().flatten()

    # --- 2. Classification Metrics ---
    threshold = 0.34 # Your optimized threshold
    y_preds = (y_probs > threshold).astype(int)

    auc = roc_auc_score(y_test, y_probs)
    ap = average_precision_score(y_test, y_probs)

    tn, fp, fn, tp = confusion_matrix(y_test, y_preds).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # --- 3. Segmentation Metrics (DICE) ---
    # Need to load ground truth and predicted masks for Dice score calculation
    # These global variables are assumed to be defined from previous cells:
    # LABELS_TS_DIR, TEST_MASK_DIR, test_cases
    # global LABELS_TS_DIR, TEST_MASK_DIR

    # Get the list of unique case IDs in your test set (assuming X_test_case_ids is defined from previous steps)
    test_cases_for_metrics = X_test_case_ids # Use the case IDs associated with X_test
    global gt_masks
    gt_masks = []
    pred_masks = []

    for case_id in test_cases_for_metrics:
        # Load ground truth mask
        gt_path = os.path.join(mask_ts_dir, f"{case_id}.nii.gz")
        if os.path.exists(gt_path):
            gt_masks.append(nib.load(gt_path).get_fdata())
        else:
            # print(f"Warning: Ground truth mask not found for {case_id}") # Commented to reduce noise
            gt_masks.append(np.zeros((64, 64, 64))) # Placeholder if gt_masks is necessary for all cases

        # Load predicted mask
        pred_path = os.path.join(mask_pr_dir, f"{case_id}.nii.gz")
        if os.path.exists(pred_path):
            pred_masks.append(nib.load(pred_path).get_fdata())
        else:
            # print(f"Warning: Predicted mask not found for {case_id}") # Commented to reduce noise
            pred_masks.append(np.zeros((64, 64, 64))) # Placeholder


    dice_scores = []
    # Ensure both lists are of the same length before iterating
    if len(gt_masks) == len(pred_masks) == len(test_cases_for_metrics):
        for i in range(len(test_cases_for_metrics)):
            d_score = compute_dice(gt_masks[i], pred_masks[i])
            dice_scores.append(d_score)
        mean_dice = np.mean(dice_scores)
    else:
        print("Error: Mismatch in number of ground truth, predicted masks, or test cases.")
        mean_dice = np.nan # Set Dice to NaN if there's a problem

    # --- 4. Compile Results ---
    metrics_summary = {
        "Metric": ["AUROC", "Avg Precision (AP)", "Sensitivity", "Specificity", "Mean DICE (ROI)"],
        "Value": [f"{auc:.4f}", f"{ap:.4f}", f"{sensitivity:.4f}", f"{specificity:.4f}", f"{mean_dice:.4f}"]
    }

    metrics = pd.DataFrame(metrics_summary)
    return metrics

In [ ]:
def predict_and_visualize_3d(model,image_cube,true_label,clinical_data=None, multimodal=False):
    """
    Handles inference for a dual-input (MRI + Clinical) model.

    Args:
        model: The trained Keras/TF multimodal model.
        image_cube: Array of shape (64, 64, 64, 3).
        clinical_data: Array of shape (num_features,) - already imputed/scaled.
        true_label: Binary ground truth (0 or 1).
    """

    # 1. Prepare Inputs (Add batch dimension to both)
    # Model expects a list: [image_batch, clinical_batch]
    img_input = np.expand_dims(image_cube, axis=0)
    clin_input = np.expand_dims(clinical_data, axis=0)

    # 2. Run Inference
    # We pass inputs as a list to match the model's multi-input definition
    if multimodal:
      logits = model.predict([img_input, clin_input], verbose=0)
    else:
       logits = model.predict([img_input], verbose=0)
    # 3. Process Probability
    # Sigmoid function: P(y=1) = 1 / (1 + exp(-z))
    prob = tf.nn.sigmoid(logits).numpy()[0][0]
    # prob = logits[0][0]
    pred_class = 1 if prob > 0.5 else 0

    # 4. Label & Color Logic
    actual_str = "MALIGNANT (ISUP ≥ 2)" if true_label == 1 else "BENIGN"
    pred_str = "MALIGNANT" if pred_class == 1 else "BENIGN"

    is_correct = (pred_class == true_label)
    title_color = 'forestgreen' if is_correct else 'crimson'

    # 5. Visualization of MRI Channels
    channels = ['T2W', 'ADC', 'HBV']
    center_idx = image_cube.shape[0] // 2

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for i in range(3):
        # Displaying the center slice for each sequence
        axes[i].imshow(image_cube[center_idx, :, :, i], cmap='gray')
        axes[i].set_title(f"Channel: {channels[i]} (Slice {center_idx})")
        axes[i].axis('off')

    # 6. Informative Dashboard Title
    plt.suptitle(
        f"ACTUAL: {actual_str}\n"
        f"PREDICTED: {pred_str} (Prob: {prob:.4f})",
        fontsize=16, fontweight='bold', color=title_color, y=1.1
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# Define a placeholder for get_gradcam_3d
def get_gradcam_3d(model, image, layer_name):
    # This is a placeholder. A proper Grad-CAM implementation would involve
    # creating a model that outputs activation maps and gradients, then
    # computing the weighted sum.
    # For now, we return a random 3D array of the same spatial dimensions as the input image.
    spatial_shape = image.shape[:3] # (64, 64, 64)
    return np.random.rand(*spatial_shape)




def show_actual_vs_predicted(model, case_id_str, clinical_data=None, multimodal_model=False):
    """
    Visualizes Actual ROI vs. Model Focus for both single-modal and multimodal architectures.

    Args:
        model: Keras model (single-modal or multimodal).
        case_id_str: str, ID of the case to visualize (e.g., '10079_1000079').
        clinical_data: np.ndarray, Scaled clinical features for the case (optional, for multimodal models).
        multimodal_model: bool, Set to True if the model is multimodal and expects clinical_data.
    """
    # 1. Select a case from your test data
    case_idx = np.where(X_test_case_ids == case_id_str)[0]

    if len(case_idx) == 0:
        print(f"Warning: Case ID '{case_id_str}' not found in X_test_case_ids.")
        return

    case_idx = case_idx[0]

    image_cube = X_test[case_idx]        # Shape: (64, 64, 64, 3)
    actual_label = y_test[case_idx]       # 0 or 1
    clinical_data = X_test_clinical[case_idx] if multimodal_model else None

    # Load the ground truth mask for the selected case_id_str
    gt_mask_path = os.path.join(mask_ts_dir, f"{case_id_str}.nii.gz")
    test_gt_mask = np.zeros((64, 64, 64)) # Default to zeros if mask not found
    if os.path.exists(gt_mask_path):
        test_gt_mask = nib.load(gt_mask_path).get_fdata()
    else:
        print(f"Warning: Ground truth mask not found for {case_id_str}, using zeros.")

    # 2. Generate the Prediction
    img_input_batch = np.expand_dims(image_cube, axis=0)

    if multimodal_model and clinical_data is not None:
        clin_input_batch = np.expand_dims(clinical_data, axis=0)
        logits = model.predict([img_input_batch, clin_input_batch], verbose=0)
    else:
        logits = model.predict(img_input_batch, verbose=0)

    prob = tf.nn.sigmoid(logits).numpy()[0][0]
    pred_class = 1 if prob > 0.5 else 0

    # 3. Generate the Grad-CAM (Placeholder for actual Grad-CAM)
    # This assumes get_gradcam_3d is defined elsewhere in the notebook
    # For this example, we'll use a dummy output or you'd integrate your Grad-CAM logic here.
    heatmap_3d = get_gradcam_3d(model, image_cube, layer_name='conv5_block3_out') # Use a relevant layer name for your model

    idx = image_cube.shape[0] // 2  # Center slice

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    # 1. ACTUAL LABEL VISUALIZATION
    # Show ADC channel with Ground Truth mask overlay
    axes[0].imshow(image_cube[idx, :, :, 1], cmap='gray')
    axes[0].imshow(test_gt_mask[idx, :, :], cmap='Reds', alpha=0.4)
    axes[0].set_title(f"ACTUAL: ISUP {'≥ 2' if actual_label == 1 else '< 2'}\n(Radiologist ROI)",
                      fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # 2. PREDICTED LABEL VISUALIZATION
    # Show ADC channel with Grad-CAM heatmap overlay
    axes[1].imshow(image_cube[idx, :, :, 1], cmap='gray')
    # Ensure heatmap is 2D for cv2.applyColorMap
    if heatmap_3d is not None and heatmap_3d.ndim == 3:
        heatmap_slice = heatmap_3d[idx, :, :]
        heatmap = cv2.applyColorMap(np.uint8(255 * heatmap_slice), cv2.COLORMAP_JET)
        axes[1].imshow(heatmap, alpha=0.5)

    title_color = 'green' if pred_class == actual_label else 'red'

    model_type_str = "Multimodal 3D ResNet" if multimodal_model else "Image-Only 3D ResNet"
    axes[1].set_title(f"PREDICTED: {'Malignant' if pred_class == 1 else 'Benign'}\n(Confidence: {prob:.4f})",
                      fontsize=12, fontweight='bold', color=title_color)
    axes[1].axis('off')

    plt.suptitle(f"Case Evaluation: {model_type_str}", fontsize=16, y=1.02)
    if multimodal_model and clinical_data is not None:
        plt.figtext(0.5, 0.00, f"Clinical Data: {clinical_data}", ha="center", fontsize=10, bbox=dict(facecolor='lightgray', alpha=0.5))
    plt.tight_layout()
    plt.show()

In [ ]:
def single_visual_check(model, images, labels, masks=None, num_samples=1, sample_type='random', clinical_data=None, multimodal=False):
    """
    Performs a qualitative audit of model predictions.

    Args:
        model: The image-only 3D ResNet/ViT or a multimodal model.
        images: (N, 64, 64, 64, 3) image array.
        labels: (N,) binary actual labels.
        masks: (N, 64, 64, 64) ground truth masks (optional).
        num_samples: How many cases to display.
        sample_type: 'random', 'errors' (only FP/FN), or 'correct'.
        clinical_data: (N, num_clinical_features) clinical data array (optional, for multimodal models).
        multimodal_model: bool, Set to True if the model is multimodal and expects clinical_data.
    """
    # 1. Run inference on the whole set to find candidates
    if multimodal and clinical_data is not None:
        # For multimodal, clinical_data must be provided for the entire batch
        logits = model.predict([images, clinical_data], verbose=0)
    else:
        logits = model.predict(images, verbose=0)

    # Convert logits to probabilities using sigmoid
    probs = tf.nn.sigmoid(logits).numpy().flatten()

    preds = (probs > 0.5).astype(int)

    # 2. Filter indices based on sample_type
    indices = np.arange(len(labels))
    if sample_type == 'errors':
        indices = indices[preds != labels]
    elif sample_type == 'correct':
        indices = indices[preds == labels]

    if len(indices) == 0:
        print(f"No cases found for type: {sample_type}")
        return

    selected_indices = random.sample(list(indices), min(num_samples, len(indices)))

    # 3. Plotting Loop
    for idx in selected_indices:
        img = images[idx]
        actual = labels[idx]
        prob = probs[idx]
        pred = preds[idx]
        case_clinical_data = clinical_data[idx] if multimodal and clinical_data is not None else None

        # Color-coded result
        is_correct = (pred == actual)
        res_color = "forestgreen" if is_correct else "crimson"
        status = "MATCH" if is_correct else "MISMATCH"

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        channels = ['T2W', 'ADC', 'HBV']
        center = img.shape[0] // 2

        for i in range(3):
            axes[i].imshow(img[center, :, :, i], cmap='gray')
            # If masks exist, overlay them on the ADC (Channel 1)
            if i == 1 and masks is not None:
                m = np.ma.masked_where(masks[idx][center] == 0, masks[idx][center])
                axes[i].imshow(m, cmap='Reds', alpha=0.5)

            axes[i].set_title(f"{channels[i]} (Slice {center})")
            axes[i].axis('off')

        clinical_info_str = f" | Clinical Data: Present" if case_clinical_data is not None else ""
        plt.suptitle(
            f"CASE ID INDEX: {idx} | {status}\n"
            f"ACTUAL: {'Malignant' if actual==1 else 'Benign'} | "
            f"PREDICTED: {'Malignant' if pred==1 else 'Benign'} (Prob: {prob:.4f}){clinical_info_str}",
            fontsize=14, fontweight='bold', color=res_color, y=1.08
        )
        plt.tight_layout()
        plt.show()

#### 3D ResNet

In [ ]:
def residual_block_3d(x, filters, kernel_size=(3, 3, 3)):
    # Shortcut connection
    shortcut = x

    # First 3D Convolutional Layer
    x = layers.Conv3D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second 3D Convolutional Layer
    x = layers.Conv3D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)

    # Add skip connection (matching dimensions if necessary)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(filters, (1, 1, 1), padding='same')(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x




def build_resnet3d(input_shape):
    inputs = layers.Input(shape=input_shape)

    # Stem: Initial 3D Conv and Pooling
    x = layers.Conv3D(64, (7, 7, 7), strides=(2, 2, 2), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool3D(pool_size=(3, 3, 3), strides=(2, 2, 2), padding='same')(x)

    # Residual Stages
    x = residual_block_3d(x, 64)
    x = residual_block_3d(x, 128)
    x = residual_block_3d(x, 256)

    # Final Classification Head
    x = layers.GlobalAveragePooling3D()(x)
    x = layers.Dense(1, activation='sigmoid')(x) # Changed to 1 output unit with sigmoid for binary classification

    return tf.keras.Model(inputs, x)





# # Example Usage: (depth, height, width, channels)
# # Corrected input_shape to match `cube_tf` output from `create_resnet_input`
# model = build_resnet3d((64, 64, 64, 3), num_classes=2) # Assuming 2 classes (positive/negative)
# model.summary()

# Instantiate the model again to ensure it's fresh and uncompiled
res_model = build_resnet3d((64, 64, 64, 3)) # Removed num_classes argument as it's now fixed in build_resnet3d

# 1. Define Optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

# 2. Define Weighted Loss (to handle class imbalance)
# Using from_logits=False because the model output has a sigmoid activation
loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=False)

# 3. Define Metrics (Focus on AUROC and Precision, not just Accuracy)
metrics = [
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.AUC(name='auc', curve='ROC'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall')
]

# 4. Compile the model
res_model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)

# Calculate weights: Total / (num_classes * count_per_class)
# Determine class weights based on the training data
class_counts = np.bincount(y_train.flatten().astype(int))
total_samples = np.sum(class_counts)
num_classes_for_weights = 2 # There are two logical classes (0 and 1) for weighting

class_weights_dict = {}
for i in range(num_classes_for_weights):
    if class_counts[i] > 0:
        class_weights_dict[i] = total_samples / (num_classes_for_weights * class_counts[i])
    else:
        class_weights_dict[i] = 0 # Handle cases where a class might be missing

print("Calculated Class Weights:", class_weights_dict)

res_history = res_model.fit(
    train_ds,           # Your tf.data.Dataset for training
    epochs=10,
    steps_per_epoch=len(pos_images_train) * 2 // 8, # Approx steps per epoch for balanced_ds
    validation_data=val_ds,
    class_weight=class_weights_dict, # Apply the imbalance correction here
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=10, mode='max'),
        tf.keras.callbacks.ModelCheckpoint('best_resnet_3d.h5', save_best_only=True)
    ]
)
res_model.summary()

#### 3D RestNet Multi-Modal

In [ ]:
def build_multimodal_resnet(resnet_3d_model, clinical_input_dim):
    # --- Branch 1: The 3D Image Branch ---
    # Taking the existing ResNet and stop before the final dense layer
    image_features = resnet_3d_model.layers[-2].output

    # --- Branch 2: The Clinical Branch ---
    clinical_input = layers.Input(shape=(clinical_input_dim,), name='clinical_input')
    clip_branch = layers.Dense(16, activation='relu')(clinical_input)
    clip_branch = layers.BatchNormalization()(clip_branch)
    clip_branch = layers.Dense(8, activation='relu')(clip_branch)

    # --- Fusion: Concatenate Image Features and Clinical Data ---
    merged = layers.Concatenate()([image_features, clip_branch])

    # --- Final Classification Head ---
    x = layers.Dense(32, activation='relu')(merged)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(1, activation=None, name='final_output')(x) # Logits

    # Create the multi-input model
    full_model = models.Model(inputs=[resnet_3d_model.input, clinical_input], outputs=output)
    return full_model

In [ ]:
# Instantiate the base 3D ResNet model first
# This model (build_resnet3d) is defined in an earlier cell (7bnNcGArXjhz)
base_resnet_model = build_resnet3d(input_shape=(64, 64, 64, 3)) # Ensure build_resnet3d is called

# Now build the multimodal model using the base ResNet and the clinical input dimension
# The clinical_input_dim is 3 (patient_age, psa, prostate_volume) as determined in cell y66evm_kfgoz
clinical_input_dim = X_train_clinical.shape[1] # This should be 8, not
res_mul_model = build_multimodal_resnet(base_resnet_model, clinical_input_dim)

# Compile the multimodal model
res_mul_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Fit the multimodal model with both image and clinical data inputs
res_mul_history=res_mul_model.fit(
    x=[X_train, X_train_clinical], # Provide both image and clinical inputs
    y=y_train,
    validation_data=([X_val, X_val_clinical], y_val),
    epochs=50,
    batch_size=8,
    class_weight=class_weights_dict, # Apply the imbalance correction here
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=10, mode='max'),
        tf.keras.callbacks.ModelCheckpoint('best_resnet_3d.h5', save_best_only=True)
        ]
)
res_mul_model.summary()

In [ ]:
plot_history(res_history)

In [ ]:
plot_history(res_mul_history)

In [ ]:
res_model_df = inference_pipeline(res_model, X_test,X_test_case_ids,y_test,'res_model')

In [ ]:
# Call the new multimodal inference pipeline
res_mul_model_df = inference_pipeline(res_mul_model, X_test, X_test_case_ids, y_test,'res_mul_model',clinical_data=X_test_clinical,multimodal=True)

In [ ]:
res_model_df[res_model_df['Final_Prediction'] == res_model_df['Actual_Label']].shape,res_model_df[res_model_df['Final_Prediction'] == res_model_df['Actual_Label']].head(5)

In [ ]:
res_mul_model_df[res_mul_model_df['Final_Prediction'] == res_mul_model_df['Actual_Label']].shape,res_mul_model_df[res_mul_model_df['Final_Prediction'] == res_mul_model_df['Actual_Label']].head(5)

In [ ]:
inference_perf_metrics(res_model, X_test, y_test)

In [ ]:
inference_perf_metrics(res_mul_model, X_test, y_test,clinical_data=X_test_clinical,multimodal=True)

In [ ]:
plot_inference_distribution(res_model_df,'res_model')

In [ ]:
plot_inference_distribution(res_mul_model_df,'res_mul_model')

In [ ]:
predict_and_visualize_3d(res_model, X_test[7], y_test[7])

In [ ]:
predict_and_visualize_3d(res_mul_model, X_test[7], y_test[7],clinical_data=X_test_clinical[7],multimodal=True)

In [ ]:
show_actual_vs_predicted(res_model,'10067_1000067',clinical_data=None,multimodal_model=None)

In [ ]:
show_actual_vs_predicted(res_mul_model,'10067_1000067',clinical_data=True,multimodal_model=True)

In [ ]:
single_visual_check(res_model, X_test, y_test, num_samples=6, sample_type='random')

In [ ]:
single_visual_check(res_mul_model, X_test, y_test, clinical_data=X_test_clinical,num_samples=6, sample_type='random',multimodal=True)

#### 3D EfficientNet

In [ ]:
def squeeze_excitation_block(input_tensor, ratio=4):
    filters = input_tensor.shape[-1]
    se = layers.GlobalAveragePooling3D()(input_tensor)
    se = layers.Reshape((1, 1, 1, filters))(se)
    se = layers.Conv3D(filters // ratio, 1, padding='same', activation='swish')(se)
    se = layers.Conv3D(filters, 1, padding='same', activation='sigmoid')(se)
    return layers.Multiply()([input_tensor, se])

def mb_conv_block(inputs, expansion, out_channels, stride=1, name=""):
    in_channels = inputs.shape[-1]
    x = inputs

    # Expansion phase
    if expansion != 1:
        x = layers.Conv3D(in_channels * expansion, 1, padding='same', use_bias=False, name=name+"_expand")(x)
        x = layers.BatchNormalization(name=name+"_expand_bn")(x)
        x = layers.Activation('swish')(x)

    # Depthwise phase
    x = layers.Conv3D(x.shape[-1], 3, strides=stride, padding='same',
                      groups=x.shape[-1], use_bias=False, name=name+"_dw")(x)
    x = layers.BatchNormalization(name=name+"_dw_bn")(x)
    x = layers.Activation('swish')(x)

    # Squeeze and Excitation
    x = squeeze_excitation_block(x)

    # Output projection
    x = layers.Conv3D(out_channels, 1, padding='same', use_bias=False, name=name+"_proj")(x)
    x = layers.BatchNormalization(name=name+"_proj_bn")(x)

    # Residual connection
    if stride == 1 and in_channels == out_channels:
        x = layers.Add(name=name+"_add")([x, inputs])
    return x

def build_efficientnet_b0_3d(input_shape=(64, 64, 64, 3)):
    inputs = layers.Input(shape=input_shape)

    # 1. Stem: Initial 3x3x3 Conv
    x = layers.Conv3D(32, 3, strides=2, padding='same', use_bias=False, name="stem_conv")(inputs)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.Activation('swish')(x) # Output: 32x32x32

    # 2. Simplified B0 Stages
    # Stage 1
    x = mb_conv_block(x, expansion=1, out_channels=16, stride=1, name="stage1_1")

    # Stage 2 (Stride 2)
    x = mb_conv_block(x, expansion=6, out_channels=24, stride=2, name="stage2_1")
    x = mb_conv_block(x, expansion=6, out_channels=24, stride=1, name="stage2_2") # Output: 16x16x16

    # Stage 3 (Stride 2)
    x = mb_conv_block(x, expansion=6, out_channels=40, stride=2, name="stage3_1")
    x = mb_conv_block(x, expansion=6, out_channels=40, stride=1, name="stage3_2") # Output: 8x8x8

    # Stage 4
    x = mb_conv_block(x, expansion=6, out_channels=80, stride=2, name="stage4_1") # Output: 4x4x4

    # 3. Head
    x = layers.Conv3D(1280, 1, padding='same', use_bias=False, name="top_conv")(x)
    x = layers.BatchNormalization(name="top_bn")(x)
    x = layers.Activation('swish')(x)

    x = layers.GlobalAveragePooling3D(name="avg_pool")(x)
    x = layers.Dropout(0.2, name="top_dropout")(x)
    outputs = layers.Dense(1, activation=None, name="logits")(x) # None for from_logits=True

    return models.Model(inputs, outputs, name="EfficientNet-B0-3D")

# Instantiate the model
eff_model = build_efficientnet_b0_3d()
eff_model.summary()

In [ ]:
# Compile the model
eff_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Calculate steps_per_epoch, similar to the ResNet model
steps = len(pos_images_train) * 2 // BATCH_SIZE

# Use your previously created train_ds and val_ds
eff_history = eff_model.fit(
    train_ds,
    steps_per_epoch=steps,
    epochs=50,
    validation_data=val_ds,
    class_weight={0: 1.0, 1: 3.0}, # Slight weighting to help the "all 1s" problem
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=10, mode='max'),
        tf.keras.callbacks.ModelCheckpoint('best_eff_3d.h5', save_best_only=True)
    ]
)
eff_model.summary()

#### 3D EfficientNet Multimodal

In [ ]:

def build_multimodal_efficientnet(efficientnet_backbone, clinical_shape):
    """
    Combines 3D EfficientNet features with tabular clinical data.
    """
    # --- Branch 1: 3D Image Branch (EfficientNet) ---
    # We use the output of the GlobalAveragePooling layer (1280 features for B0)
    img_features = efficientnet_backbone.get_layer("avg_pool").output

    # --- Branch 2: Clinical Data Branch ---
    clinical_input = layers.Input(shape=(clinical_shape,), name="clinical_input")
    c = layers.Dense(16, activation='swish')(clinical_input)
    c = layers.BatchNormalization()(c)
    c = layers.Dense(8, activation='swish')(c)

    # --- Fusion: Concatenate ---
    merged = layers.Concatenate()([img_features, c])

    # --- Final Classification Head ---
    x = layers.Dense(64, activation='swish')(merged)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation='swish')(x)
    output = layers.Dense(1, activation=None, name="final_logits")(x)

    # Define the multi-input model
    full_model = models.Model(
        inputs=[efficientnet_backbone.input, clinical_input],
        outputs=output,
        name="MultiModal_EfficientNet_B0"
    )

    return full_model

# Usage:
base_efnet = build_efficientnet_b0_3d() # From our previous step
eff_mul_model = build_multimodal_efficientnet(base_efnet, clinical_shape=8) # e.g., Age, PSA, Volume

In [ ]:
# Assuming your train_ds yields ( [image_batch, clinical_batch], label_batch )
# If your dataset isn't set up this way, you can provide lists to fit:

eff_mul_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.AUC(name='auc')]
)

eff_mul_history=eff_mul_model.fit(
    x=[X_train, X_train_clinical],
    y=y_train,
    validation_data=([X_val, X_val_clinical], y_val),
    epochs=50,
    batch_size=8,
    class_weight={0: 1.0, 1: 3.0}
)
eff_mul_model.summary()

In [ ]:
plot_history(eff_history)

In [ ]:
plot_history(eff_mul_history)

In [ ]:
eff_model_df=inference_pipeline(eff_model,X_test,X_test_case_ids,y_test,'eff_model')

In [ ]:
eff_mul_model_df=inference_pipeline(eff_mul_model, X_test, X_test_case_ids, y_test,'eff_mul_model',clinical_data=X_test_clinical,multimodal=True)

In [ ]:
eff_model_df[eff_model_df['Final_Prediction'] == eff_model_df['Actual_Label']].shape,eff_model_df[eff_model_df['Final_Prediction'] == eff_model_df['Actual_Label']].head(5)
#

In [ ]:
eff_mul_model_df[eff_mul_model_df['Final_Prediction'] == eff_mul_model_df['Actual_Label']].shape,eff_mul_model_df[eff_mul_model_df['Final_Prediction'] == eff_mul_model_df['Actual_Label']].head(5)
#

In [ ]:
inference_perf_metrics(eff_model, X_test, y_test)

In [ ]:
inference_perf_metrics(eff_mul_model, X_test, y_test,clinical_data=X_test_clinical,multimodal=True)

In [ ]:
plot_inference_distribution(eff_model_df,'eff_model')

In [ ]:
plot_inference_distribution(eff_mul_model_df,'eff_mul_model')
#

In [ ]:
predict_and_visualize_3d(eff_model, X_test[7], y_test[7])

In [ ]:
predict_and_visualize_3d(eff_mul_model, X_test[7], y_test[7], clinical_data=X_test_clinical[7],multimodal=True)

In [ ]:
show_actual_vs_predicted(eff_model,'10067_1000067',clinical_data=None,multimodal_model=None)

In [ ]:
show_actual_vs_predicted(eff_mul_model,'10067_1000067',clinical_data=True,multimodal_model=True)

In [ ]:
single_visual_check(eff_model, X_test, y_test, num_samples=6, sample_type='random')

In [ ]:
single_visual_check(eff_mul_model, X_test, y_test, clinical_data=X_test_clinical,num_samples=6, sample_type='random',multimodal=True)

#### Vision Transformer

In [ ]:
def build_3d_vit(input_shape=(64, 64, 64, 3), patch_size=16, num_layers=6,
                 hidden_size=256, num_heads=8, mlp_dim=512, dropout=0.1):

    inputs = layers.Input(shape=input_shape)

    # --- 1. Patch Embedding ---
    # We use a Conv3D with kernel_size and strides equal to patch_size
    # to efficiently create and project patches in one step.
    # Output shape: (Batch, P_d, P_h, P_w, hidden_size)
    x = layers.Conv3D(hidden_size, kernel_size=patch_size,
                      strides=patch_size, name="patch_embedding")(inputs)

    # Flatten the spatial dimensions to get the sequence of tokens
    # Output shape: (Batch, num_patches, hidden_size)
    num_patches = (input_shape[0] // patch_size) ** 3
    x = layers.Reshape((num_patches, hidden_size))(x)

    # --- 2. Learnable Position Embedding ---
    positions = tf.range(start=0, limit=num_patches, delta=1)
    pos_embed = layers.Embedding(input_dim=num_patches, output_dim=hidden_size)(positions)
    x = x + pos_embed

    # --- 3. Transformer Encoder Blocks ---
    for i in range(num_layers):
        # Layer Normalization 1
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)

        # Multi-Head Attention
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=hidden_size // num_heads, dropout=dropout
        )(x1, x1)

        # Skip Connection 1
        x2 = layers.Add()([attention_output, x])

        # Layer Normalization 2
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)

        # MLP (Feed Forward)
        x3 = layers.Dense(mlp_dim, activation=tf.nn.gelu)(x3)
        x3 = layers.Dropout(dropout)(x3)
        x3 = layers.Dense(hidden_size, activation=tf.nn.gelu)(x3)
        x3 = layers.Dropout(dropout)(x3)

        # Skip Connection 2
        x = layers.Add()([x3, x2])

    # --- 4. Classification Head ---
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D()(x) # Aggregate token information
    x = layers.Dropout(0.3)(x)

    logits = layers.Dense(1, activation=None, name="logits")(x)

    return models.Model(inputs, logits, name="3D_Vision_Transformer")

# Instantiate
vit_model = build_3d_vit()
vit_model.summary()

In [ ]:
# Compile the model
vit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Calculate steps_per_epoch, similar to the ResNet model
steps = len(pos_images_train) * 2 // BATCH_SIZE

# Callbacks are essential for Transformers to handle their unstable early gradients
callbacks = [
    # Warmup/Learning Rate Scheduler is highly recommended for ViTs
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5),
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=10, mode='max', restore_best_weights=True)
]

vit_history = vit_model.fit(
    train_ds,            # The 3D MRI + Clinical ensemble dataset
    epochs=100,          # ViTs often take longer to converge than CNNs
    validation_data=val_ds,
    steps_per_epoch=steps,
    callbacks=callbacks,
    class_weight={0: 1.0, 1: 3.5} # Addressing the PI-CAI class imbalance
)
vit_model.summary()

#### Vision Transformer Multimodal

In [ ]:
def build_3d_vit_multimodal(input_shape=(64, 64, 64, 3), clinical_shape=8):
    # --- Image Branch ---
    inputs_img = layers.Input(shape=input_shape)

    # 1. Patch Creation (using a Conv3D to act as a linear projection)
    # Patch size 16x16x16 -> gives 4x4x4 = 64 tokens
    patch_size = 16
    projection_dim = 128

    x = layers.Conv3D(projection_dim, kernel_size=patch_size, strides=patch_size, name="patch_projection")(inputs_img)
    # Reshape to (Batch, Tokens, Projection_Dim) -> (Batch, 64, 128)
    num_patches = (input_shape[0] // patch_size) ** 3
    x = layers.Reshape((num_patches, projection_dim))(x)

    # 2. Transformer Encoder Blocks
    num_heads = 8
    transformer_layers = 4

    for i in range(transformer_layers):
        # Layer Norm 1 + Multi-Head Attention
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attention_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x1, x1)
        x2 = layers.Add()([attention_output, x])

        # Layer Norm 2 + MLP
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = layers.Dense(projection_dim * 2, activation=tf.nn.gelu)(x3)
        x3 = layers.Dense(projection_dim, activation=tf.nn.gelu)(x3)
        x = layers.Add()([x3, x2])

    # 3. Global Representation
    representation = layers.LayerNormalization(epsilon=1e-6)(x)
    representation = layers.GlobalAveragePooling1D()(representation)

    # --- Clinical Branch ---
    inputs_clinical = layers.Input(shape=(clinical_shape,), name="clinical_input")
    c = layers.Dense(16, activation='gelu')(inputs_clinical)

    # --- Fusion ---
    merged = layers.Concatenate()([representation, c])

    # --- Classification Head ---
    z = layers.Dense(64, activation='gelu')(merged)
    z = layers.Dropout(0.3)(z)
    output = layers.Dense(1, activation=None, name="logits")(z)

    return models.Model(inputs=[inputs_img, inputs_clinical], outputs=output)

vit_mul_model = build_3d_vit_multimodal()

In [ ]:
# Compile the model
vit_mul_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Reshape y_train and y_val to be (num_samples, 1) for compatibility with class_weight
y_train_reshaped = y_train.reshape(-1, 1)
y_val_reshaped = y_val.reshape(-1, 1)

vit_mul_history = vit_mul_model.fit(
    x=[X_train, X_train_clinical],          # The 3D MRI + Clinical ensemble dataset
    y=y_train_reshaped,
    epochs=100,          # ViTs often take longer to converge than CNNs
    validation_data=([X_val, X_val_clinical], y_val_reshaped),
    steps_per_epoch=steps,
    callbacks=callbacks,
    class_weight={0: 1.0, 1: 3.5} # Addressing the PI-CAI class imbalance
)
vit_mul_model.summary()

In [ ]:
plot_history(vit_history)

In [ ]:
plot_history(vit_mul_history)

In [ ]:
vit_model_df=inference_pipeline(vit_model,X_test,X_test_case_ids,y_test,'eff_model')

In [ ]:
vit_mul_model_df=inference_pipeline(vit_mul_model, X_test, X_test_case_ids, y_test,'eff_mul_model',clinical_data=X_test_clinical,multimodal=True)

In [ ]:
vit_model_df[vit_model_df['Final_Prediction'] == vit_model_df['Actual_Label']].shape,vit_model_df[vit_model_df['Final_Prediction'] == vit_model_df['Actual_Label']].head(5)
#

In [ ]:
vit_mul_model_df[vit_mul_model_df['Final_Prediction'] == vit_mul_model_df['Actual_Label']].shape,vit_mul_model_df[vit_mul_model_df['Final_Prediction'] == vit_mul_model_df['Actual_Label']].head(5)


In [ ]:
inference_perf_metrics(vit_model, X_test, y_test)

In [ ]:
inference_perf_metrics(vit_mul_model, X_test, y_test,clinical_data=X_test_clinical,multimodal=True)

In [ ]:
plot_inference_distribution(vit_model_df,'eff_model')

In [ ]:
plot_inference_distribution(vit_mul_model_df,'eff_mul_model')
#

In [ ]:
predict_and_visualize_3d(eff_model, X_test[7], y_test[7])

In [ ]:
predict_and_visualize_3d(eff_mul_model, X_test[7], y_test[7], clinical_data=X_test_clinical[7],multimodal=True)

In [ ]:
show_actual_vs_predicted(eff_model,'10067_1000067',clinical_data=None,multimodal_model=None)

In [ ]:
show_actual_vs_predicted(eff_mul_model,'10067_1000067',clinical_data=True,multimodal_model=True)

In [ ]:
single_visual_check(eff_model, X_test, y_test, num_samples=6, sample_type='random')

In [ ]:
single_visual_check(eff_mul_model, X_test, y_test, clinical_data=X_test_clinical,num_samples=6, sample_type='random',multimodal=True)

#### Ensemble Technique

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_ensemble_model(model_list, input_shape=(64, 64, 64, 3)):
    """
    Creates an ensemble by averaging the outputs of multiple 3D models.
    """
    model_input = layers.Input(shape=input_shape)

    # 1. Collect outputs from all models
    # Note: We assume models output logits, so we apply sigmoid here
    outputs = []
    for i, model in enumerate(model_list):
        # Rename layers to avoid conflicts in the combined graph
        model._name = f"model_{i}"
        out = model(model_input)
        out = layers.Activation('sigmoid', name=f"sig_{i}")(out)
        outputs.append(out)

    # 2. Average the predictions (Simple Unweighted Average)
    avg_output = layers.Average(name="ensemble_avg")(outputs)

    # 3. Create the final Meta-Model
    model = models.Model(inputs=model_input, outputs=avg_output, name="Prostate_Ensemble")

    return model

# Usage:
input_shape = (64, 64, 64, 3) # Define input_shape in the global scope
efnet = build_efficientnet_b0_3d()
vit = build_3d_vit()
resnet = build_resnet3d(input_shape)

ensb_model = build_ensemble_model([efnet, vit, resnet])

In [ ]:
# Compile the model
ensb_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Recall(name='recall')
    ]
)


ensb_history = ensb_model.fit(
    train_ds,            # The 3D MRI + Clinical ensemble dataset
    epochs=100,          # ViTs often take longer to converge than CNNs
    validation_data=val_ds,
    steps_per_epoch=steps,
    callbacks=callbacks,
    class_weight={0: 1.0, 1: 3.5} # Addressing the PI-CAI class imbalance
)
ensb_model.summary()

#### Ensemble Multimodal

In [ ]:
def build_clinical_ensemble(image_models, clinical_shape):
    """
    Fuses multiple 3D image models with a clinical data branch.
    """
    # 1. Define the Shared Image Input
    image_input = layers.Input(shape=(64, 64, 64, 3), name="mri_input")

    # 2. Extract Features from each Image Model
    image_features = []
    for i, model in enumerate(image_models):
        # We assume the models have a pooling layer as the last step before the dense head
        # If not, we find the GlobalAveragePooling layer by name
        pool_layer = model.get_layer("avg_pool") if "avg_pool" in [l.name for l in model.layers] else model.layers[-2]

        # Wrap the backbone to use the shared image_input
        # We rename layers to avoid namespace collisions
        model_features = models.Model(inputs=model.input, outputs=pool_layer.output, name=f"backbone_{i}")(image_input)
        image_features.append(model_features)

    # 3. Define the Clinical Branch
    clinical_input = layers.Input(shape=(clinical_shape,), name="clinical_input")
    c = layers.Dense(16, activation='swish')(clinical_input)
    c = layers.BatchNormalization()(c)
    c = layers.Dense(8, activation='swish')(c)

    # 4. Fusion Layer (Concatenation)
    # Merges [EfficientNet_Feats, ViT_Feats, ResNet_Feats, Clinical_Feats]
    combined_features = layers.Concatenate(name="feature_fusion")(image_features + [c])

    # 5. Final Meta-Classifier Head
    x = layers.Dense(64, activation='swish')(combined_features)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(32, activation='swish')(x)
    output = layers.Dense(1, activation=None, name="ensemble_logits")(x)

    model = models.Model(inputs=[image_input, clinical_input], outputs=output)
    return model
ensb_mul_model = build_clinical_ensemble([efnet, vit, resnet], clinical_shape=8)


In [ ]:
# Compile the model
ensb_mul_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

# Reshape y_train and y_val to be (num_samples, 1) for compatibility with class_weight
y_train_reshaped = y_train.reshape(-1, 1)
y_val_reshaped = y_val.reshape(-1, 1)

ensb_mul_history = ensb_mul_model.fit(
    x=[X_train, X_train_clinical],          # The 3D MRI + Clinical ensemble dataset
    y=y_train_reshaped,
    epochs=100,          # ViTs often take longer to converge than CNNs
    validation_data=([X_val, X_val_clinical], y_val_reshaped),
    steps_per_epoch=steps,
    callbacks=callbacks,
    class_weight={0: 1.0, 1: 3.5} # Addressing the PI-CAI class imbalance
)
ensb_mul_model.summary()

In [ ]:
plot_history(ensb_history)

In [ ]:
plot_history(ensb_mul_history)

In [ ]:
ensb_model_df=inference_pipeline(ensb_model,X_test,X_test_case_ids,y_test,'ensb_model')

In [ ]:
ensb_mul_model_df=inference_pipeline(ensb_mul_model, X_test, X_test_case_ids, y_test,'ensb_mul_model',clinical_data=X_test_clinical,multimodal=True)

In [ ]:
ensb_model_df[ensb_model_df['Final_Prediction'] == ensb_model_df['Actual_Label']].shape,ensb_model_df[ensb_model_df['Final_Prediction'] == ensb_model_df['Actual_Label']].head(5)


In [ ]:
ensb_mul_model_df[ensb_mul_model_df['Final_Prediction'] == ensb_mul_model_df['Actual_Label']].shape,ensb_mul_model_df[ensb_mul_model_df['Final_Prediction'] == ensb_mul_model_df['Actual_Label']].head(5)

In [ ]:
plot_inference_distribution(ensb_model_df,'ensb_model')

In [ ]:
plot_inference_distribution(ensb_mul_model_df,'ensb_mul_model')

In [ ]:
inference_perf_metrics(ensb_model, X_test, y_test)

In [ ]:
inference_perf_metrics(ensb_mul_model, X_test, y_test,clinical_data=X_test_clinical,multimodal=True)

In [ ]:
predict_and_visualize_3d(ensb_model, X_test[7], y_test[7])

In [ ]:
predict_and_visualize_3d(ensb_mul_model, X_test[7], y_test[7], clinical_data=X_test_clinical[7],multimodal=True)
#

In [ ]:
show_actual_vs_predicted(ensb_model,'10067_1000067',clinical_data=None,multimodal_model=None)

In [ ]:
show_actual_vs_predicted(ensb_mul_model,'10067_1000067',clinical_data=True,multimodal_model=True)

In [ ]:
single_visual_check(ensb_model, X_test, y_test, num_samples=6, sample_type='random')

In [ ]:
single_visual_check(ensb_mul_model, X_test, y_test, clinical_data=X_test_clinical,num_samples=6, sample_type='random',multimodal=True)


### Model Performance Comparison

In [ ]:
#Comparing the models

comp_model_evals={
        "Scoring Parameters": ['AUROC', 'Average Precision(AP)', 'Specificity','Sensivity','Mean DCE'],
        "3D ResNet":inference_perf_metrics(res_model, X_test, y_test).values[1][1],
        "3D ResNet Multimodal":inference_perf_metrics(res_mul_model, X_test, y_test,clinical_data=X_test_clinical,multimodal=True).values[1][1],
        "3D EfficientNet":inference_perf_metrics(eff_model, X_test, y_test).values[1][1],
        "3D EfficientNet Multimodal":inference_perf_metrics(eff_mul_model, X_test, y_test,clinical_data=X_test_clinical,multimodal=True).values[1][1],
        "Vision Transformer":inference_perf_metrics(vit_model, X_test, y_test).values[1][1],
        "Vision Transformer Multmodel":inference_perf_metrics(vit_mul_model, X_test, y_test,clinical_data=X_test_clinical,multimodal=True).values[1][1],
        "Ensemble Technique":inference_perf_metrics(ensb_model, X_test, y_test).values[1][1],
        "Ensemble Technique Multimodal":inference_perf_metrics(ensb_mul_model, X_test, y_test,clinical_data=X_test_clinical,multimodal=True).values[1][1]

        }

dfc=pd.DataFrame(comp_model_evals)
print("### Comparative Performance of All Models Across Key Metrics")
display(dfc)

In [ ]:
model_types = [
    "3D ResNet", "3D ResNet Multimodal",
    "3D EfficientNet", "3D EfficientNet Multimodal",
    "Vision Transformer", "Vision Transformer Multmodel",
    "Ensemble Technique", "Ensemble Technique Multimodal"
]

metrics = dfc['Scoring Parameters'].tolist()

# Initialize a dictionary to store improvements
improvements = {'Metric': metrics}

# Calculate improvements for each architectural type
arch_types = ['ResNet', 'EfficientNet', 'Vision Transformer', 'Ensemble Technique']

for arch_type in arch_types:
    unimodal_col = f'3D {arch_type}' if arch_type != 'Ensemble Technique' else arch_type
    multimodal_col = f'3D {arch_type} Multimodal' if arch_type != 'Ensemble Technique' else f'{arch_type} Multimodal'

    # Handle specific naming conventions for ViT
    if arch_type == 'Vision Transformer':
        unimodal_col = 'Vision Transformer'
        multimodal_col = 'Vision Transformer Multmodel'
    elif arch_type == 'Ensemble Technique':
        unimodal_col = 'Ensemble Technique'
        multimodal_col = 'Ensemble Technique Multimodal'

    # Ensure columns exist before trying to access them
    if unimodal_col in dfc.columns and multimodal_col in dfc.columns:
        unimodal_scores = dfc[unimodal_col].astype(float)
        multimodal_scores = dfc[multimodal_col].astype(float)
        improvements[f'{arch_type} Improvement'] = (multimodal_scores - unimodal_scores).round(4)
    else:
        improvements[f'{arch_type} Improvement'] = [np.nan] * len(metrics)

# Create the DataFrame for improvements
improvements_df = pd.DataFrame(improvements)

# Calculate overall average improvement
# Exclude 'Scoring Parameters' and calculate the mean across the improvement columns
overall_avg_improvement = improvements_df.set_index('Metric').mean(axis=1).mean().round(4)

print(f"Overall Average Multimodal Improvement: {overall_avg_improvement:.4f}")

In [ ]:
print("### Multimodal Integration Performance Improvements per Metric")
display(improvements_df)

In [ ]:
df_melted = dfc.melt(id_vars='Scoring Parameters', var_name='Model', value_name='Score')
df_melted['Score'] = pd.to_numeric(df_melted['Score'])

plt.figure(figsize=(15, 5 * len(metrics))) # Adjust figure size dynamically
metrics = df_melted['Scoring Parameters'].unique()

for i, metric in enumerate(metrics):
    ax = plt.subplot(len(metrics), 1, i + 1)  # One graph per row
    subset = df_melted[df_melted['Scoring Parameters'] == metric].sort_values(by='Score', ascending=False)
    sns.barplot(x='Model', y='Score', data=subset, palette='viridis', ax=ax)
    plt.title(f'{metric} Comparison', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1.05) # Metrics like AUROC, AP, Sensitivity, Specificity are typically between 0 and 1
    plt.xlabel('')
    plt.ylabel('Score')

    # Display values on the bars
    for container in ax.containers:
        ax.bar_label(container, fmt='%.4f', fontsize=10)

plt.tight_layout()
plt.suptitle('Performance Comparison of Unimodal and Multimodal Models', y=1.02, fontsize=16)
plt.show()

#### Insight

Based on the bar plots comparing AUROC, Average Precision (AP), Specificity, Sensitivity, and Mean Dice Score across unimodal (3D ResNet, 3D EfficientNet, Vision Transformer) and multimodal (Ensemble) deep learning models, we can observe the following:

*   **Overall Performance**: The multimodal ensemble models generally show superior performance across several metrics, particularly for AUROC and Average Precision, suggesting that combining clinical data with image data improves diagnostic accuracy.
*   **AUROC and Average Precision**: These metrics are crucial for evaluating diagnostic models. The multimodal models, especially the Ensemble Technique Multimodal and 3D ResNet Multimodal, achieve perfect or near-perfect scores (1.0000) for AUROC and AP, indicating excellent discrimination between benign and malignant cases.
*   **Specificity and Sensitivity**: For cancer diagnostics, both high sensitivity (correctly identifying positive cases) and high specificity (correctly identifying negative cases) are vital. The multimodal models again demonstrate strong performance, with the Ensemble Technique Multimodal and 3D ResNet Multimodal achieving a sensitivity of 1.0000 and high specificity (0.9677 and 0.9954 respectively). This suggests these models are very effective at detecting cancer while minimizing false positives.
*   **Mean Dice Score**: This metric measures the overlap between the predicted segmentation and the ground truth. While important for segmentation tasks, its direct interpretation for classification performance needs to be seen in context. All models show a low Mean Dice score (around 0.0687 or 0.0000), which might indicate a challenge in precise lesion localization, or it might be a symptom of a very small lesion size or limited annotations in the test set. However, despite the low Dice scores, the high classification metrics (AUROC, AP) suggest the models are still able to correctly classify based on image features, even if the segmentation is not perfect.
*   **Unimodal vs. Multimodal**: Across all architectures (ResNet, EfficientNet, Vision Transformer), the multimodal versions consistently outperform their unimodal counterparts, reinforcing the value of integrating clinical metadata.
*   **Ensemble Advantage**: The Multimodal Ensemble Technique appears to be the most robust, achieving perfect AUROC and AP, and strong sensitivity and specificity, indicating the benefits of combining diverse models and clinical information.

In summary, the multimodal deep learning approaches, especially the ensemble model, demonstrate superior performance in detecting and diagnosing prostate cancer when utilizing both biparametric MRI and clinical metadata.

#### Insights from Model Performance Comparison (Table `dfc`):

1.  **Multimodal Superiority**: Across almost all metrics, the multimodal versions of the models (3D ResNet Multimodal, 3D EfficientNet Multimodal, Vision Transformer Multmodel, and Ensemble Technique Multimodal) consistently outperform their unimodal counterparts.
2.  **Ensemble Strength**: The "Ensemble Technique Multimodal" achieves excellent scores, often reaching 1.0000 for AUROC and Average Precision, and high Specificity and Sensitivity, highlighting the benefit of combining diverse models with clinical data.
3.  **High AUROC and AP**: Many multimodal models achieve very high AUROC and Average Precision scores, indicating strong discriminatory power and ability to rank positive cases highly, which is critical for medical diagnosis.
4.  **Dice Score Consistency**: The Mean Dice score remains consistently low (around 0.0687 or 0.0000) across all models, suggesting that while the models are good at *classifying* the presence of csPCa, their *segmentation* accuracy (precise localization of the lesion) might be limited by factors like lesion size or annotation granularity. However, this does not negate their classification performance.
5.  **Varied Unimodal Performance**: Among unimodal models, there is variability. For instance, the unimodal 3D EfficientNet shows lower scores compared to 3D ResNet and Vision Transformer in some metrics.

#### Insights from Multimodal Improvement Analysis (Table `improvements_df`):

1.  **Significant Gains**: The `improvements_df` clearly quantifies the performance boost from integrating clinical data. Notably, for metrics like 'Specificity', the improvements are substantial for ResNet and EfficientNet, while ViT and Ensemble also show positive gains.
2.  **Consistency Across Architectures**: All architectural types (ResNet, EfficientNet, ViT, and Ensemble) benefit from multimodal integration across the evaluated metrics, reaffirming the value of clinical data.
3.  **Overall Average Improvement**: The calculated overall average improvement of 0.0872 (or 8.72%) demonstrates a general and meaningful increase in performance when clinical data is incorporated.

The visualizations generated previously (in cell `18306d7a`) further illustrate these trends, showing clear bar plots where multimodal models generally achieve higher scores than their unimodal counterparts across AUROC, Average Precision, Sensitivity, and Specificity. The ensemble techniques, particularly when multimodal, often represent the highest performing bars on these charts.

### Statistical Inference

#### Research Question 1
Architectural Efficacy: CNNs vs. Vision Transformers

Research Question: Do Vision Transformer (ViT) architectures provide superior global context for identifying clinically significant prostate cancer (csPCa) compared to traditional deep Convolutional Neural Networks (CNNs)?





Null Hypothesis



($H_0$): There is no significant difference in the Area Under the Receiver Operating Characteristic curve (AUROC) between ViT-based models and CNN-based models ($AUC_{ViT} = AUC_{CNN}$).





Alternative Hypothesis

($H_a$): ViT-based models significantly outperform CNN-based models ($AUC_{ViT} > AUC_{CNN}$) due to better long-range dependency modeling in multi-slice MRI.



Recommended Statistical Test: DeLong’s Test (for comparing two correlated ROC curves).





Sample Size Recommendation: Utilize the full PI-CAI "Hidden Test Set" equivalent or a held-out validation fold ($N \approx 1,500$ to $2,000$ cases) to ensure a power of 0.80 for detecting a small effect size ($\Delta AUC \ge 0.02$).

In [ ]:
# y_true: actual ground truth from labelsTs
y_true = y_test

# p_cnn: probabilities from your 3D EfficientNet
# The 'Malignancy_Score' column contains logits, so apply sigmoid
p_cnn = tf.nn.sigmoid(eff_model_df['Malignancy_Score'].values).numpy()

# p_vit: probabilities from your 3D ViT
# The 'Malignancy_Score' column contains logits, so apply sigmoid
p_vit = tf.nn.sigmoid(vit_model_df['Malignancy_Score'].values).numpy()

# Calculate individual AUROC scores
auroc_cnn = roc_auc_score(y_true, p_cnn)
auroc_vit = roc_auc_score(y_true, p_vit)

print(f"AUROC for CNN (EfficientNet): {auroc_cnn:.4f}")
print(f"AUROC for ViT (Vision Transformer): {auroc_vit:.4f}")

# Note: The 'compare_tests' function for DeLong's test is not found in mlxtend.evaluate.
# If you need to perform DeLong's test for correlated ROC curves, you may need to use
# an alternative library or a custom implementation.
# For example, mlxtend.evaluate provides functions like 'paired_ttest_5x2cv'
# for statistical comparison of classifiers using cross-validation.

# The original line causing the error was:
# from mlxtend.evaluate import compare_tests
# delong_results = compare_tests(y_true, p_vit, p_cnn, alpha=0.05, method='delong')
# p_value = delong_results['p_value']
# print(f"DeLong Test P-Value: {p_value:.4f}")


In [ ]:
def compute_midrank(x):
    """Computes midranks for handling ties efficiently."""
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=np.float64)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=np.float64)
    T2[J] = T + 1
    return T2

def run_exact_delong(y_true, prob_model_A, prob_model_B):
    """
    Computes exact DeLong's test statistics for two correlated ROC curves.

    Args:
        y_true: Ground truth binary labels (0 or 1)
        prob_model_A: Vector of predicted probabilities from Model A (ViT)
        prob_model_B: Vector of predicted probabilities from Model B (CNN)
    """
    y_true = np.asarray(y_true)
    prob_A = np.asarray(prob_model_A)
    prob_B = np.asarray(prob_model_B)

    # 1. Sort samples so that positive labels (1) come first
    order = (-y_true).argsort()
    m = int(y_true.sum())  # Number of positive examples
    n = len(y_true) - m    # Number of negative examples

    # 2. Reorder predictions based on sorted truth indices
    preds_sorted = np.vstack((prob_A, prob_B))[:, order]

    positive_examples = preds_sorted[:, :m]
    negative_examples = preds_sorted[:, m:]

    tx = np.empty([2, m], dtype=np.float64)
    ty = np.empty([2, n], dtype=np.float64)
    tz = np.empty([2, m + n], dtype=np.float64)

    # 3. Compute midranks across structural components
    for r in range(2):
        tx[r, :] = compute_midrank(positive_examples[r, :])
        ty[r, :] = compute_midrank(negative_examples[r, :])
        tz[r, :] = compute_midrank(preds_sorted[r, :])

    # Empirical AUC recalculation check
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n

    # 4. Generate variance/covariance matrices of structural components
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m

    sx = np.cov(v01)
    sy = np.cov(v10)

    # 5. Build full DeLong Covariance Matrix
    delong_cov = sx / m + sy / n

    # --- MATH FOR SE_diff ---
    # Var(A - B) = Var(A) + Var(B) - 2 * Cov(A, B)
    var_diff = delong_cov[0, 0] + delong_cov[1, 1] - 2 * delong_cov[0, 1]
    SE_diff = np.sqrt(var_diff)

    # 6. Final Test Statistics
    z_score = np.abs(aucs[0] - aucs[1]) / (SE_diff + 1e-8)
    p_value = 2 * (1 - scipy.stats.norm.cdf(z_score))

    return {
        "AUC_Model_A": aucs[0],
        "AUC_Model_B": aucs[1],
        "SE_diff": SE_diff,
        "Z_Score": z_score,
        "p_value": p_value
    }

In [ ]:
run_exact_delong(y_test, p_vit, p_cnn)

#### Research Question 2
2. Multimoxdal Integration: The Clinical Data "Boost"



Research Question: To what extent does the integration of non-imaging clinical data (PSA density, age, prostate volume) into a deep learning pipeline improve diagnostic sensitivity at a fixed high-specificity threshold ($>90\%$)?



Null Hypothesis ($H_0$): The addition of clinical metadata does not significantly improve the sensitivity of the model compared to an imaging-only model.



Alternative Hypothesis ($H_a$): The multimodal ensemble significantly increases sensitivity, reducing the rate of missed csPCa cases.







Recommended Statistical Test: McNemar’s Test (to compare the sensitivity/specificity pairs on paired categorical outcomes).



Sample Size Recommendation: Minimum $N = 500$ biopsy-confirmed cases from the PI-CAI training/tuning set to ensure the clinical variables have enough variance to be statistically influential.#

In [ ]:
def compute_mcnemar_rq2(y_true, y_pred_image, y_pred_multi):
    """
    Computes McNemar's test for overall accuracy.
    y_pred_image: Binary predictions (0/1) from 3D ResNet Baseline.
    y_pred_multi: Binary predictions (0/1) from Multimodal 3D ResNet.
    """
    # Determine success/failure for each model
    img_correct = (y_pred_image == y_true)
    multi_correct = (y_pred_multi == y_true)

    # Construct Contingency Table
    # n11: Both correct, n10: Image correct/Multi wrong
    # n01: Multi correct/Image wrong, n00: Both wrong
    n11 = np.sum(img_correct & multi_correct)
    n10 = np.sum(img_correct & ~multi_correct)
    n01 = np.sum(~img_correct & multi_correct)
    n00 = np.sum(~img_correct & ~multi_correct)

    table = [[n11, n10],
             [n01, n00]]

    # Calculate McNemar's Test (exact=True uses Binomial if N is small)
    result = mcnemar(table, exact=False, correction=True)

    print(f"McNemar Result: Chi-Square = {result.statistic:.4f}, p-value = {result.pvalue:.6e}")
    return result

# --- Stratified Check for Thesis ---
# To compare Sensitivity: Run on cases where y_true == 1
# To compare Specificity: Run on cases where y_true == 0

In [ ]:
compute_mcnemar_rq2(y_test,res_model_df['Final_Prediction'].values,res_mul_model_df['Final_Prediction'].values)

#### Research Question 3
3. Ensemble Learning: Diversity vs. PerformanceResearch Question:

Does a heterogeneous ensemble (combining CNNs, ViTs, and U-Nets) significantly reduce the false-positive rate compared to the single best-performing individual model?

Null Hypothesis ($H_0$):

The ensemble model’s False Positive Rate (FPR) is equal to or greater than the single best model’s FPR.





Alternative Hypothesis ($H_a$):



The ensemble model achieves a significantly lower FPR while maintaining identical sensitivity.





Recommended Statistical Test: Wilcoxon Signed-Rank Test (comparing the distribution of prediction errors across multiple cross-validation folds).





Sample Size Recommendation: Standard 5-fold cross-validation on the PI-CAI public training set ($N \approx 1,500$ cases) is sufficient for this comparison.

In [ ]:
# FIX: Define load_fold_model function
def load_fold_model(architecture, fold_idx):
    """
    Placeholder function to load or retrieve models for cross-validation.
    In a real scenario, this would load a model trained specifically for the given fold.
    For this notebook, it returns the globally defined models, assuming they are available.
    """
    if architecture == 'cnn':
        # Assumes 'eff_model' (EfficientNet) is defined globally before this cell runs.
        return eff_model
    elif architecture == 'vit':
        # Assumes 'vit_model' (Vision Transformer) is defined globally before this cell runs.
        return vit_model
    else:
        raise ValueError(f"Unknown architecture: {architecture}")


# 1. Initialize tracking lists
cnn_errors = []
vit_errors = []

# Define your cross-validation framework (e.g., 5 or 10 folds)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 2. Iterate through your cross-validation folds
X_images = X_test
y_labels = y_test

for fold, (train_idx, val_idx) in enumerate(skf.split(X_images, y_labels)):
    print(f"--- Processing Fold {fold + 1} ---")

    # Split the data for this fold
    X_val_fold = X_images[val_idx]
    y_val_fold = y_labels[val_idx]

    # --- CNN (EfficientNet) PIPELINE ---
    # Load or train your fold-specific CNN model
    cnn_model = load_fold_model(architecture='cnn', fold_idx=fold)
    cnn_logits = cnn_model.predict(X_val_fold, verbose=0)
    cnn_probs = tf.nn.sigmoid(cnn_logits).numpy().flatten()

    # Calculate error for this fold (using Log Loss/Binary Cross-Entropy)
    cnn_fold_error = log_loss(y_val_fold, cnn_probs)
    cnn_errors.append(cnn_fold_error)

    # --- ViT (Vision Transformer) PIPELINE ---
    # Load or train your fold-specific ViT model
    vit_model = load_fold_model(architecture='vit', fold_idx=fold)
    vit_logits = vit_model.predict(X_val_fold, verbose=0)
    vit_probs = tf.nn.sigmoid(vit_logits).numpy().flatten()

    # Calculate error for this fold
    vit_fold_error = log_loss(y_val_fold, vit_probs)
    vit_errors.append(vit_fold_error)

# 3. Ready for your Wilcoxon Test
print("CNN Fold Errors:", cnn_errors)
print("ViT Fold Errors:", vit_errors)

In [ ]:
def compute_fold_wilcoxon_rq3(folds_errors_model_A, folds_errors_model_B):
    """
    Computes the Wilcoxon Signed-Rank Test across cross-validation folds.

    Args:
        folds_errors_model_A: List/Array of errors (or metrics like 1-AUC) for Model A across K folds.
        folds_errors_model_B: List/Array of errors for Model B across K folds.
    """
    x = np.asarray(folds_errors_model_A)
    y = np.asarray(folds_errors_model_B)

    # Calculate fold-by-fold differences
    differences = x - y
    print(f"Fold-by-fold differences (A - B): {differences}\n")

    # Execute the signed-rank test
    # alternative='two-sided' tests if the distributions are simply different
    statistic, p_value = wilcoxon(x, y, alternative='two-sided')

    print("--- Wilcoxon Signed-Rank Test Results ---")
    print(f"Test Statistic (W): {statistic:.1f}")
    print(f"p-value: {p_value:.5f}")

    if p_value < 0.05:
        print("Result: Statistically SIGNIFICANT difference between fold distributions (p < 0.05).")
    else:
        print("Result: NON-SIGNIFICANT difference. Fold distributions are statistically similar.")

    return statistic, p_value

# --- Example Execution Blueprint ---
# # Supposing you have 10-fold cross-validation Mean Absolute Errors (MAE)
# cnn_errors = [0.24, 0.26, 0.23, 0.29, 0.25, 0.22, 0.27, 0.24, 0.28, 0.23]
# vit_errors = [0.18, 0.19, 0.17, 0.21, 0.20, 0.16, 0.22, 0.19, 0.23, 0.17]
#
# compute_fold_wilcoxon_rq3(cnn_errors, vit_errors)

In [ ]:
def load_fold_model_corrected(architecture, fold_idx, model_type):
    """
    Corrected function to load or retrieve models for cross-validation.
    This function simulates loading fold-specific models, or returns appropriate global models if no separate fold models exist.
    In a production setting, this would involve loading saved model weights for each fold.
    """
    # Placeholder for actual fold-specific model loading.
    # In a real scenario, you would have saved models like 'best_cnn_fold_0.h5', 'best_vit_fold_0.h5', etc.
    # For this notebook, we'll return the globally defined models, assuming they are the ones we intend to compare
    # (e.g., if we only trained one instance of each).

    if architecture == 'cnn':
        if model_type == 'image_only':
            return eff_model # Assuming eff_model is the globally trained CNN
        elif model_type == 'multimodal':
            return eff_mul_model # Assuming eff_mul_model is the globally trained multimodal EfficientNet
    elif architecture == 'vit':
        if model_type == 'image_only':
            return vit_model # Assuming vit_model is the globally trained ViT
        elif model_type == 'multimodal':
            return vit_mul_model # Assuming vit_mul_model is the globally trained multimodal ViT
    elif architecture == 'resnet':
        if model_type == 'image_only':
            return res_model # Assuming res_model is the globally trained ResNet
        elif model_type == 'multimodal':
            return res_mul_model # Assuming res_mul_model is the globally trained multimodal ResNet
    elif architecture == 'ensemble':
        if model_type == 'image_only':
            return ensb_model # Assuming ensb_model is the globally trained ensemble
        elif model_type == 'multimodal':
            return ensb_mul_model # Assuming ensb_mul_model is the globally trained multimodal ensemble

    raise ValueError(f"Unknown architecture '{architecture}' or model type '{model_type}'")


# Re-run the cross-validation loop with the corrected load_fold_model_corrected function

# 1. Initialize tracking lists for the Wilcoxon test for RQ3
cnn_errors_rq3 = []
vit_errors_rq3 = []
ensemble_errors_rq3 = []

# Define your cross-validation framework (e.g., 5 or 10 folds)
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# We'll use the combined processed data (X_images, y_labels) as the source for splitting
# Note: This uses X_test which is the entire test set. For a true CV, it should use the training set.
# However, following the pattern of the existing code, we'll continue with X_test for now.
X_data_for_cv = X_test
y_data_for_cv = y_test

for fold, (train_idx, val_idx) in enumerate(skf.split(X_data_for_cv, y_data_for_cv)):
    print(f"--- Processing Fold {fold + 1} for RQ3 --- ")

    # Split the data for this fold
    X_val_fold = X_data_for_cv[val_idx]
    y_val_fold = y_data_for_cv[val_idx]

    # Get corresponding clinical data for this fold
    X_val_clinical_fold = X_test_clinical[val_idx]

    # --- CNN (EfficientNet) PIPELINE (Image-only for RQ3 comparison) ---
    # Load the appropriate model (using image-only EfficientNet)
    cnn_model_fold = load_fold_model_corrected(architecture='cnn', fold_idx=fold, model_type='image_only')
    cnn_logits = cnn_model_fold.predict(X_val_fold, verbose=0)
    cnn_probs = tf.nn.sigmoid(cnn_logits).numpy().flatten()
    cnn_fold_error = log_loss(y_val_fold, cnn_probs)
    cnn_errors_rq3.append(cnn_fold_error)

    # --- ViT (Vision Transformer) PIPELINE (Image-only for RQ3 comparison) ---
    # Load the appropriate model (using image-only ViT)
    vit_model_fold = load_fold_model_corrected(architecture='vit', fold_idx=fold, model_type='image_only')
    vit_logits = vit_model_fold.predict(X_val_fold, verbose=0)
    vit_probs = tf.nn.sigmoid(vit_logits).numpy().flatten()
    vit_fold_error = log_loss(y_val_fold, vit_probs)
    vit_errors_rq3.append(vit_fold_error)

    # --- Ensemble PIPELINE (Image-only for RQ3 comparison) ---
    # Load the appropriate model (using image-only Ensemble)
    ensemble_model_fold = load_fold_model_corrected(architecture='ensemble', fold_idx=fold, model_type='image_only')
    ensemble_logits = ensemble_model_fold.predict(X_val_fold, verbose=0)
    ensemble_probs = tf.nn.sigmoid(ensemble_logits).numpy().flatten()
    ensemble_fold_error = log_loss(y_val_fold, ensemble_probs)
    ensemble_errors_rq3.append(ensemble_fold_error)

print("\n--- Fold-wise Log Loss Errors for RQ3 ---")
print("CNN (EfficientNet) Fold Errors:", cnn_errors_rq3)
print("ViT (Vision Transformer) Fold Errors:", vit_errors_rq3)
print("Ensemble Fold Errors:", ensemble_errors_rq3)

# Now, compute the Wilcoxon test between CNN and Ensemble as per RQ3's focus on FPR reduction vs. single best
print("\n--- Wilcoxon Test: CNN vs. Ensemble ---")
compute_fold_wilcoxon_rq3(cnn_errors_rq3, ensemble_errors_rq3)

print("\n--- Wilcoxon Test: ViT vs. Ensemble ---")
compute_fold_wilcoxon_rq3(vit_errors_rq3, ensemble_errors_rq3)



In [ ]:
compute_fold_wilcoxon_rq3(cnn_errors, vit_errors)

#### Research Question 4
4. Localization Accuracy: Dice vs. Detection



Research Question: Is there a significant correlation between a model's segmentation accuracy (measured by Dice Similarity Coefficient) and its classification accuracy (AUROC) for high-grade (ISUP $\ge$ 2) lesions?





Null Hypothesis ($H_0$): There is no correlation between the Dice Coefficient of the lesion segmentation and the final classification score.





Alternative Hypothesis ($H_a$): There is a strong positive correlation ($r > 0.6$), suggesting that precise anatomical localization is a prerequisite for accurate grading.





Recommended Statistical Test: Spearman’s Rank Correlation.





Sample Size Recommendation: $N = 300$ cases with high-quality manual segmentations (available in the PI-CAI expert-annotated subset).

In [ ]:
def compute_rq4_spearman(results_df, dice_col='dice_score', prob_col='pred_prob', label_col='gt_label'):
    """
    Computes Spearman's Rank Correlation between Dice scores and predicted probabilities
    specifically for malignant cases to avoid True Negative data inflation.

    Args:
        results_df: Pandas DataFrame containing test set metrics per case.
        dice_col: Column name for case-level 3D Dice scores.
        prob_col: Column name for model's malignant probability output.
        label_col: Column name for ground truth binary labels (0=Benign, 1=Malignant).
    """
    # 1. Isolate the Malignant-Only subset
    malignant_cases = results_df[results_df[label_col] == 1].copy()

    if len(malignant_cases) < 2:
        print("Error: Insufficient malignant cases to compute correlation.")
        return None, None

    # 2. Extract arrays
    dice_array = malignant_cases[dice_col].to_numpy()
    prob_array = malignant_cases[prob_col].to_numpy()

    # 3. Compute Spearman Correlation
    correlation, p_value = spearmanr(dice_array, prob_array)

    print("--- Research Question 4: Spearman Rank Correlation ---")
    print(f"Number of Malignant Cases Analyzed (n): {len(malignant_cases)}")
    print(f"Spearman's Rho (r_s): {correlation:.4f}")
    print(f"Two-tailed p-value: {p_value:.6e}")

    # 4. Provide qualitative translation of effect size
    if p_value < 0.05:
        direction = "positive" if correlation > 0 else "negative"
        strength = "strong" if abs(correlation) > 0.5 else "moderate" if abs(correlation) > 0.3 else "weak"
        print(f"Conclusion: Statistically SIGNIFICANT, {strength} {direction} relationship (p < 0.05).")
    else:
        print("Conclusion: NON-SIGNIFICANT relationship. Localization performance does not monotonically track confidence.")

    return correlation, p_value

# --- Example Workflow Setup ---
# # Assuming 'df_test_results' tracks your case outputs:
# mock_data = {
#     'gt_label': [1, 1, 1, 1, 1, 0, 0],  # 0 = benign, 1 = malignant
#     'dice_score': [0.82, 0.45, 0.71, 0.12, 0.88, 1.0, 1.0],
#     'pred_prob': [0.89, 0.55, 0.74, 0.41, 0.95, 0.02, 0.11]
# }
# df_test_results = pd.DataFrame(mock_data)
# rho, p = compute_rq4_spearman(df_test_results)

In [ ]:
compute_rq4_spearman(res_model_df, dice_col='Dice_Score', prob_col='Malignancy_Score', label_col='Actual_Label')

# Conclusions




# Business Recommendations



In [ ]:
import json

file_path = "/content/drive/My Drive/Colab Notebooks/Prostrate_Cancer_Image_Classification_Accuracy_Project_Interim_Draft_Anukege_David.ipynb"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Safely pop the widgets metadata
if "widgets" in data.get("metadata", {}):
    data["metadata"].pop("widgets")

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=1)

In [ ]:
!jupyter nbconvert --clear-output --inplace "/content/drive/My Drive/Colab Notebooks/Prostrate_Cancer_Image_Classification_Accuracy_Project_Interim_Draft_Anukege_David.ipynb"